In [7]:
# Cell 1 — Setup paths + hard checks + hardened OUT_DIR + file hashes

import os
import shutil
import json
import hashlib
import numpy as np
import pandas as pd

from google.colab import drive

# Clean old mountpoint if needed
if os.path.exists("/content/drive") and os.path.isdir("/content/drive"):
    try:
        if os.listdir("/content/drive"):   # only if non-empty
            shutil.rmtree("/content/drive")
    except Exception:
        pass

drive.mount("/content/drive", force_remount=True)

DATA_DIR = "/content/drive/MyDrive/EXPORT_FOR_COLAB"
X_FP = os.path.join(DATA_DIR, "X_samples_x_genes.tsv.gz")
Y_FP = os.path.join(DATA_DIR, "y_labels.tsv")
META_FP = os.path.join(DATA_DIR, "meta.tsv")

OUT_DIR = os.path.join(DATA_DIR, "ML_RESULTS_UPGRADED")

def ensure_out_dir(out_dir):
    os.makedirs(out_dir, exist_ok=True)
    if not os.path.isdir(out_dir):
        raise RuntimeError(f"OUT_DIR is not a directory: {out_dir}")
    test_fp = os.path.join(out_dir, "_write_test.tmp")
    try:
        with open(test_fp, "w") as f:
            f.write("ok")
        os.remove(test_fp)
    except Exception as e:
        raise RuntimeError(
            f"Cannot write to OUT_DIR (Drive might be unmounted): {out_dir}\n{e}"
        )

ensure_out_dir(OUT_DIR)

def md5_file(fp, block=2**20):
    h = hashlib.md5()
    with open(fp, "rb") as f:
        while True:
            b = f.read(block)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

for fp in [X_FP, Y_FP, META_FP]:
    if not os.path.exists(fp):
        raise FileNotFoundError(f"Missing required file: {fp}")

print("✅ Inputs found")
print("✅ OUT_DIR:", OUT_DIR)
print("MD5 X:", md5_file(X_FP))
print("MD5 y:", md5_file(Y_FP))
print("MD5 meta:", md5_file(META_FP))

Mounted at /content/drive
✅ Inputs found
✅ OUT_DIR: /content/drive/MyDrive/EXPORT_FOR_COLAB/ML_RESULTS_UPGRADED
MD5 X: ca81cbe775e0b1a9bd2dec0eebd015ea
MD5 y: ab48dafaa60ceb42f8aa8060f5c4f07a
MD5 meta: d418f6072ec3e533837f05ede853b62d


In [8]:
# ============================================
# Cell 2 — Load data with strict alignment
# ============================================

y_df = pd.read_csv(Y_FP, sep="\t")
if not {"SAMPLE_ID","y"}.issubset(y_df.columns):
    raise ValueError(f"y_labels.tsv must contain SAMPLE_ID and y. Found: {list(y_df.columns)}")
if not y_df["SAMPLE_ID"].is_unique:
    raise ValueError("Duplicate SAMPLE_ID in y_labels.tsv")

meta_df = pd.read_csv(META_FP, sep="\t")
if "SAMPLE_ID" not in meta_df.columns:
    raise ValueError("meta.tsv must contain SAMPLE_ID")

X_df = pd.read_csv(X_FP, sep="\t", compression="gzip")
if "SAMPLE_ID" not in X_df.columns:
    raise ValueError("X_samples_x_genes.tsv.gz must contain SAMPLE_ID")
if not X_df["SAMPLE_ID"].is_unique:
    raise ValueError("Duplicate SAMPLE_ID in X")

df = X_df.merge(y_df, on="SAMPLE_ID", how="inner")
if df.shape[0] != y_df.shape[0]:
    missing = set(y_df["SAMPLE_ID"]) - set(X_df["SAMPLE_ID"])
    raise ValueError(f"Some labels missing in X. Missing count={len(missing)}")

if df["y"].isna().any():
    raise ValueError("Missing y after merge (should not happen).")

print("✅ df:", df.shape)
print("Class counts:\n", df["y"].value_counts())

✅ df: (1974, 20387)
Class counts:
 y
LumA           700
LumB           475
Her2           224
Claudin-low    218
Basal          209
Normal         148
Name: count, dtype: int64


In [9]:

# Cell 3 — Robust preprocessing (log detector) + matrices


from sklearn.preprocessing import LabelEncoder

feature_cols = [c for c in df.columns if c not in ("SAMPLE_ID", "y")]
X_mat = df[feature_cols].to_numpy(dtype=np.float32)

le6 = LabelEncoder()
y6 = le6.fit_transform(df["y"].astype(str))
y6_str = le6.inverse_transform(y6)

print("✅ X:", X_mat.shape)
print("✅ 6-class labels:", list(le6.classes_))

x_min, x_max = float(np.nanmin(X_mat)), float(np.nanmax(X_mat))
p_neg = float(np.mean(np.nan_to_num(X_mat) < 0))
q50 = float(np.nanpercentile(X_mat, 50))
q99 = float(np.nanpercentile(X_mat, 99))
tail_ratio = float((q99 + 1e-6) / (q50 + 1e-6))

APPLY_LOG1P = (p_neg == 0.0) and (x_max >= 500) and (tail_ratio >= 20)

print("Range:", x_min, "to", x_max)
print("Neg fraction:", p_neg)
print("q50:", q50, "q99:", q99, "tail_ratio:", tail_ratio)
print("APPLY_LOG1P:", APPLY_LOG1P)

if APPLY_LOG1P:
    X_mat = np.log1p(X_mat).astype(np.float32)
    print("✅ Applied log1p")
else:
    print("✅ Skipped log1p (already log/z-like)")

✅ X: (1974, 20385)
✅ 6-class labels: ['Basal', 'Claudin-low', 'Her2', 'LumA', 'LumB', 'Normal']
Range: 4.557910442352295 to 14.879412651062012
Neg fraction: 0.0
q50: 6.061042308807373 q99: 11.79688549041748 tail_ratio: 1.946345849942251
APPLY_LOG1P: False
✅ Skipped log1p (already log/z-like)


In [10]:

# Cell 4 — Define tasks
#   Primary: tumor-only 4-class (Basal/Her2/LumA/LumB)
#   Secondary: tumor-only 5-class; full 6-class (exploratory)


from sklearn.preprocessing import LabelEncoder


# Primary task (paper): tumor-only 4-class

PRIMARY_4CLASS_ORDER = ["Basal", "Her2", "LumA", "LumB"]  # fixed order
mask4 = np.isin(np.array(y6_str, dtype=object), np.array(PRIMARY_4CLASS_ORDER, dtype=object))

X4 = X_mat[mask4].astype(np.float32)
y4_str = np.array(y6_str, dtype=object)[mask4].astype(str)

lab_map4 = {c:i for i,c in enumerate(PRIMARY_4CLASS_ORDER)}
y4 = np.array([lab_map4[s] for s in y4_str], dtype=int)

print(" PRIMARY 4-class size:", X4.shape)
print("PRIMARY counts:", pd.Series(y4_str).value_counts().to_dict())

# Secondary tasks

stage1 = np.where(y6_str == "Normal", "Normal_like", "Tumor_rich")

tumor_mask = (y6_str != "Normal")
X_tumor = X_mat[tumor_mask]
y_tumor_str = y6_str[tumor_mask]

le5 = LabelEncoder()
y5 = le5.fit_transform(y_tumor_str)

print("Stage-1 counts:", pd.Series(stage1).value_counts().to_dict())
print("Secondary tumor-only size:", X_tumor.shape)
print("Secondary classes:", list(le5.classes_))
print("Secondary class counts:\n", pd.Series(y_tumor_str).value_counts())

 PRIMARY 4-class size: (1608, 20385)
PRIMARY counts: {'LumA': 700, 'LumB': 475, 'Her2': 224, 'Basal': 209}
Stage-1 counts: {'Tumor_rich': 1826, 'Normal_like': 148}
Secondary tumor-only size: (1826, 20385)
Secondary classes: ['Basal', 'Claudin-low', 'Her2', 'LumA', 'LumB']
Secondary class counts:
 LumA           700
LumB           475
Her2           224
Claudin-low    218
Basal          209
Name: count, dtype: int64


In [11]:

# Cell 5 — ExperimentConfig (PRIMARY = 4-class)


RANDOM_SEED = 123
np.random.seed(RANDOM_SEED)

CONFIG = {
  "dataset": "METABRIC",
  "primary_claim": {
    "task": "Tumor-only 4-class (Basal/Her2/LumA/LumB)",
    "split": "Fixed 70/15/15 per seed; 5 seeds",
    "seeds": [123,124,125,126,127],
    "primary_metrics": ["macro-F1", "bACC", "macro-ROC-AUC(ovr)", "macro-PR-AUC"],
    "reject_rule": "Choose threshold on VAL only (maximize macro-F1 with coverage>=min_cov), apply fixed threshold to TEST."
  },
  "pam50_benchmarks": [
    "PAM50 baseline A: ML-style (train-only robust scale + corr-to-centroid)",
    "PAM50 baseline B: Canonical genefu (molecular.subtyping) with recommended mapping/scaling"
  ],
  "secondary_analyses": [
    "5-class tumor-only (incl Claudin-low): reported as secondary/appendix",
    "6-class incl Normal: exploratory only",
    "TOP80 stability: descriptive gene inclusion frequency (not a locked externally validated signature)"
  ],
  "V_prefilter": 3000,
  "panel_size": 80,
  "selector": {"penalty": "l1", "solver": "saga", "C_SEL": 0.6, "max_iter": 1500, "tol": 1e-3},
  "classifier": {"penalty": "l2", "solver": "lbfgs", "C_grid": [0.3,1.0,3.0], "max_iter": 4000, "class_weight": "balanced"},
  "apply_log1p": bool(APPLY_LOG1P),
}

print(json.dumps(CONFIG, indent=2))
print("PRIMARY X4.shape =", X4.shape, "| y4.shape =", y4.shape)

{
  "dataset": "METABRIC",
  "primary_claim": {
    "task": "Tumor-only 4-class (Basal/Her2/LumA/LumB)",
    "split": "Fixed 70/15/15 per seed; 5 seeds",
    "seeds": [
      123,
      124,
      125,
      126,
      127
    ],
    "primary_metrics": [
      "macro-F1",
      "bACC",
      "macro-ROC-AUC(ovr)",
      "macro-PR-AUC"
    ],
    "reject_rule": "Choose threshold on VAL only (maximize macro-F1 with coverage>=min_cov), apply fixed threshold to TEST."
  },
  "pam50_benchmarks": [
    "PAM50 baseline A: ML-style (train-only robust scale + corr-to-centroid)",
    "PAM50 baseline B: Canonical genefu (molecular.subtyping) with recommended mapping/scaling"
  ],
  "secondary_analyses": [
    "5-class tumor-only (incl Claudin-low): reported as secondary/appendix",
    "6-class incl Normal: exploratory only",
    "TOP80 stability: descriptive gene inclusion frequency (not a locked externally validated signature)"
  ],
  "V_prefilter": 3000,
  "panel_size": 80,
  "selector": {
    "

In [12]:
# ============================================
# Cell 6 — SECONDARY (5-class tumor-only)
# split + prefilter + save raw splits + persist preprocessing artifacts
# Robust scaling with train-only median/MAD (no leakage)
# ============================================

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
import numpy as np, os

req = ["X_tumor", "y5", "feature_cols", "le5", "OUT_DIR", "RANDOM_SEED", "ensure_out_dir", "APPLY_LOG1P"]
missing = [k for k in req if k not in globals()]
if missing:
    raise NameError(f"Missing in memory: {missing}. Run Cells 1–5 first. Missing={missing}")

ensure_out_dir(OUT_DIR)
np.random.seed(int(RANDOM_SEED))

# ---- enforce secondary naming ----
X_5_full = X_tumor.astype(np.float32)
y_5 = y5.astype(int)
classes_5 = np.array(le5.classes_, dtype=str)

V_5 = int(min(CONFIG.get("V_prefilter", 3000), X_5_full.shape[1]))
print("SECONDARY 5-class | Using V_5 =", V_5)

# ---- 70/15/15 stratified split (5-class) ----
sss1_5 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=int(RANDOM_SEED))
tr_idx_5, tmp_idx_5 = next(sss1_5.split(X_5_full, y_5))

Xtr_5_raw, ytr_5 = X_5_full[tr_idx_5], y_5[tr_idx_5]
Xtmp_5_raw, ytmp_5 = X_5_full[tmp_idx_5], y_5[tmp_idx_5]

sss2_5 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=int(RANDOM_SEED))
va_rel_5, te_rel_5 = next(sss2_5.split(Xtmp_5_raw, ytmp_5))
va_idx_5 = tmp_idx_5[va_rel_5]
te_idx_5 = tmp_idx_5[te_rel_5]

Xva_5_raw, yva_5 = X_5_full[va_idx_5], y_5[va_idx_5]
Xte_5_raw, yte_5 = X_5_full[te_idx_5], y_5[te_idx_5]

print("SECONDARY split sizes:", Xtr_5_raw.shape, Xva_5_raw.shape, Xte_5_raw.shape)

# ---- train-only variance ranking ----
imp_rank_5 = SimpleImputer(strategy="median")
Xtr_5_imp_rank = imp_rank_5.fit_transform(Xtr_5_raw)
var_5 = np.var(Xtr_5_imp_rank, axis=0)
ranked_5 = np.argsort(var_5)[::-1]
v_idx_5 = ranked_5[:V_5]

Xtr_5_v = Xtr_5_raw[:, v_idx_5]
Xva_5_v = Xva_5_raw[:, v_idx_5]
Xte_5_v = Xte_5_raw[:, v_idx_5]

# ---- train-only impute ----
imp_5 = SimpleImputer(strategy="median")
Xtr_5_i = imp_5.fit_transform(Xtr_5_v)
Xva_5_i = imp_5.transform(Xva_5_v)
Xte_5_i = imp_5.transform(Xte_5_v)

# ---- robust scaling: train-only median/MAD ----
def robust_scale_train(X_train, X):
    med = np.median(X_train, axis=0).astype(np.float32)
    mad = (np.median(np.abs(X_train - med), axis=0) + 1e-6).astype(np.float32)
    return ((X - med) / mad).astype(np.float32), med, mad

Xtr_5, med_5, mad_5 = robust_scale_train(Xtr_5_i, Xtr_5_i)
Xva_5 = ((Xva_5_i - med_5) / mad_5).astype(np.float32)
Xte_5 = ((Xte_5_i - med_5) / mad_5).astype(np.float32)

selected_genes_5 = np.array(feature_cols, dtype=str)[v_idx_5]
print("SECONDARY Xtr_5/Xva_5/Xte_5:", Xtr_5.shape, Xva_5.shape, Xte_5.shape)
print("selected_genes_5 example:", selected_genes_5[:10].tolist())

# ---- persist secondary split ----
split_fp_5 = os.path.join(OUT_DIR, f"stage2_split_seed{int(RANDOM_SEED)}.npz")
np.savez(
    split_fp_5,
    tr_idx=tr_idx_5.astype(int),
    va_idx=va_idx_5.astype(int),
    te_idx=te_idx_5.astype(int),
    classes=classes_5,
    random_seed=int(RANDOM_SEED),
    V=int(V_5),
    apply_log1p=bool(APPLY_LOG1P),
)
print("✅ Saved SECONDARY Stage-2 split:", split_fp_5)

SECONDARY 5-class | Using V_5 = 3000
SECONDARY split sizes: (1278, 20385) (274, 20385) (274, 20385)
SECONDARY Xtr_5/Xva_5/Xte_5: (1278, 3000) (274, 3000) (274, 3000)
selected_genes_5 example: ['SCGB2A2', 'SCGB1D2', 'MUCL1', 'PIP', 'ANKRD30A', 'TFF1', 'HLA-DRB5', 'TFF3', 'CLEC3A', 'CPB1']
✅ Saved SECONDARY Stage-2 split: /content/drive/MyDrive/EXPORT_FOR_COLAB/ML_RESULTS_UPGRADED/stage2_split_seed123.npz


In [13]:
# ============================================
# Cell 7 — SECONDARY (5-class) fast baselines on selected V_5
# ============================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix
import numpy as np, pandas as pd, os

req = ["Xtr_5", "Xva_5", "Xte_5", "ytr_5", "yva_5", "yte_5", "OUT_DIR", "RANDOM_SEED", "classes_5"]
missing = [k for k in req if k not in globals()]
if missing:
    raise NameError(f"Missing in memory: {missing}. Run Cell 6 first.")

K_5 = len(classes_5)

def zscore_rows(X):
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True) + 1e-8
    return (X - mu) / sd

def cosine_sim(A, B):
    A2 = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-8)
    B2 = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-8)
    return A2 @ B2.T

def eval_preds(y_true, y_pred, tag=""):
    bacc = float(balanced_accuracy_score(y_true, y_pred))
    mf1  = float(f1_score(y_true, y_pred, average="macro"))
    print(f"\n[{tag}] bACC={bacc:.3f} | macro-F1={mf1:.3f}")
    print(confusion_matrix(y_true, y_pred, labels=np.arange(K_5)))
    return bacc, mf1

# A) Centroid scorer
Xtr_5_z = zscore_rows(Xtr_5)
Xva_5_z = zscore_rows(Xva_5)
Xte_5_z = zscore_rows(Xte_5)

centroids_5 = np.stack([Xtr_5_z[ytr_5 == k].mean(axis=0) for k in range(K_5)], axis=0)

va_scores_5 = cosine_sim(Xva_5_z, centroids_5)
te_scores_5 = cosine_sim(Xte_5_z, centroids_5)

va_pred_cent_5 = np.argmax(va_scores_5, axis=1)
te_pred_cent_5 = np.argmax(te_scores_5, axis=1)

eval_preds(yva_5, va_pred_cent_5, "SECONDARY Centroid/VAL")
eval_preds(yte_5, te_pred_cent_5, "SECONDARY Centroid/TEST")

# B) Logistic Regression baseline
lr_5 = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    class_weight="balanced",
    max_iter=3000,
    n_jobs=1,
    random_state=int(RANDOM_SEED)
)
lr_5.fit(Xtr_5, ytr_5)

va_pred_lr_5 = lr_5.predict(Xva_5)
te_pred_lr_5 = lr_5.predict(Xte_5)

eval_preds(yva_5, va_pred_lr_5, "SECONDARY LR/VAL")
eval_preds(yte_5, te_pred_lr_5, "SECONDARY LR/TEST")

baseline_5_df = pd.DataFrame([
    {
        "model": "centroid",
        "val_bacc": float(balanced_accuracy_score(yva_5, va_pred_cent_5)),
        "val_macro_f1": float(f1_score(yva_5, va_pred_cent_5, average="macro")),
        "test_bacc": float(balanced_accuracy_score(yte_5, te_pred_cent_5)),
        "test_macro_f1": float(f1_score(yte_5, te_pred_cent_5, average="macro")),
    },
    {
        "model": "logreg",
        "val_bacc": float(balanced_accuracy_score(yva_5, va_pred_lr_5)),
        "val_macro_f1": float(f1_score(yva_5, va_pred_lr_5, average="macro")),
        "test_bacc": float(balanced_accuracy_score(yte_5, te_pred_lr_5)),
        "test_macro_f1": float(f1_score(yte_5, te_pred_lr_5, average="macro")),
    }
])

baseline_5_fp = os.path.join(OUT_DIR, "fast_baselines_val_test_secondary_5class.csv")
baseline_5_df.to_csv(baseline_5_fp, index=False)
print("\n✅ Saved:", baseline_5_fp)
baseline_5_df


[SECONDARY Centroid/VAL] bACC=0.708 | macro-F1=0.683
[[26  1  3  0  2]
 [ 9 20  2  1  0]
 [ 0  4 23  1  5]
 [ 0 11  7 73 14]
 [ 0  6  7  8 51]]

[SECONDARY Centroid/TEST] bACC=0.723 | macro-F1=0.697
[[24  2  4  1  0]
 [13 17  0  3  0]
 [ 2  0 30  0  2]
 [ 1  4  2 82 16]
 [ 0  2 16  6 47]]

[SECONDARY LR/VAL] bACC=0.825 | macro-F1=0.829
[[27  2  1  0  2]
 [ 2 29  1  0  0]
 [ 1  2 23  2  5]
 [ 0  1  2 93  9]
 [ 0  0  3 12 57]]

[SECONDARY LR/TEST] bACC=0.844 | macro-F1=0.845
[[26  3  1  1  0]
 [ 4 26  0  3  0]
 [ 3  0 28  0  3]
 [ 0  0  2 97  6]
 [ 1  0  4  6 60]]

✅ Saved: /content/drive/MyDrive/EXPORT_FOR_COLAB/ML_RESULTS_UPGRADED/fast_baselines_val_test_secondary_5class.csv


,model,val_bacc,val_macro_f1,test_bacc,test_macro_f1
0,centroid,0.707608,0.682984,0.722924,0.696720
1,logreg,0.824870,0.828696,0.843800,0.844508


In [ ]:

# Cell 8 — SECONDARY (5-class) compact selector (tune on VAL) + exports


import numpy as np, pandas as pd, os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression

req = ["Xtr_5","Xva_5","ytr_5","yva_5","selected_genes_5","OUT_DIR","RANDOM_SEED"]
missing = [k for k in req if k not in globals()]
if missing:
    raise NameError(f"Missing in memory: {missing}. Run Cells 6–7 first.")

C_SEL_GRID_5 = [0.1, 0.3, 0.6, 1.0, 2.0]
PANEL_GRID_5 = [60, 80, 100, 120]
MAX_ITER_SEL_5 = 1500
TOL_SEL_5 = 1e-3

DO_CHEAP_STABILITY_5 = True
STAB_FOLDS_5 = 3

def fit_sparse_selector_scores(X, y, C, seed):
    m = LogisticRegression(
        penalty="l1",
        solver="saga",
        C=float(C),
        class_weight="balanced",
        max_iter=int(MAX_ITER_SEL_5),
        tol=float(TOL_SEL_5),
        n_jobs=1,
        random_state=int(seed)
    )
    m.fit(X, y)
    return np.mean(np.abs(m.coef_), axis=0)

def get_panel_from_scores(mean_score, panel_size):
    rank = np.argsort(mean_score)[::-1]
    return rank[:int(panel_size)]

def eval_panel_val_macroF1(Xtr, ytr, Xva, yva, panel_idx, seed):
    lr = LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        class_weight="balanced",
        max_iter=4000,
        n_jobs=1,
        random_state=int(seed)
    )
    lr.fit(Xtr[:, panel_idx], ytr)
    pred = lr.predict(Xva[:, panel_idx])
    return float(f1_score(yva, pred, average="macro"))

best_5 = {
    "macro_f1": -1.0,
    "C_SEL": None,
    "PANEL_SIZE": None,
    "panel_idx": None,
    "mean_score": None,
    "sd_score": None
}

for C_SEL in C_SEL_GRID_5:
    base = fit_sparse_selector_scores(Xtr_5, ytr_5, C=C_SEL, seed=RANDOM_SEED)
    scores = [base]

    if DO_CHEAP_STABILITY_5:
        skf = StratifiedKFold(n_splits=int(STAB_FOLDS_5), shuffle=True, random_state=int(RANDOM_SEED))
        for j, (itr, _) in enumerate(skf.split(Xtr_5, ytr_5), start=1):
            scores.append(
                fit_sparse_selector_scores(Xtr_5[itr], ytr_5[itr], C=C_SEL, seed=int(RANDOM_SEED) + 10*j)
            )

    scores = np.vstack(scores)
    mean_score = scores.mean(axis=0)
    sd_score = scores.std(axis=0)

    for PANEL_SIZE in PANEL_GRID_5:
        panel_idx = get_panel_from_scores(mean_score, PANEL_SIZE)
        mf1 = eval_panel_val_macroF1(Xtr_5, ytr_5, Xva_5, yva_5, panel_idx, seed=RANDOM_SEED)

        if mf1 > best_5["macro_f1"]:
            best_5.update({
                "macro_f1": mf1,
                "C_SEL": float(C_SEL),
                "PANEL_SIZE": int(PANEL_SIZE),
                "panel_idx": panel_idx.copy(),
                "mean_score": mean_score.copy(),
                "sd_score": sd_score.copy(),
            })

print("SECONDARY best selector settings (VAL macro-F1):")
print(" C_SEL_5 =", best_5["C_SEL"])
print(" PANEL_SIZE_5 =", best_5["PANEL_SIZE"])
print(" VAL macro-F1_5 =", round(best_5["macro_f1"], 4))

C_SEL_5 = best_5["C_SEL"]
PANEL_SIZE_5 = best_5["PANEL_SIZE"]
panel_local_idx_5 = best_5["panel_idx"]

panel_genes_5 = np.array(selected_genes_5, dtype=str)[panel_local_idx_5]

panel_5_df = pd.DataFrame({
    "GENE": np.array(selected_genes_5, dtype=str),
    "mean_abscoef": best_5["mean_score"],
    "sd_abscoef": best_5["sd_score"]
}).sort_values(["mean_abscoef","GENE"], ascending=[False, True])

panel_scores_5_fp = os.path.join(OUT_DIR, "panel_gene_scores_secondary_5class.csv")
panel_5_fp = os.path.join(OUT_DIR, f"compact_panel_secondary_5class_TOP{int(PANEL_SIZE_5)}.csv")

panel_5_df.to_csv(panel_scores_5_fp, index=False)
pd.Series(panel_genes_5, name="GENE").to_csv(panel_5_fp, index=False)

print("✅ Saved:", panel_scores_5_fp)
print("✅ Saved:", panel_5_fp)
print("Top genes (secondary):", panel_genes_5[:15].tolist())

In [ ]:

# Cell 9 (secondary) — Evaluate compact panel (5-class)
# FIXED: uses RANDOM_SEED (no RANDOM_SEED_5 dependency)


from sklearn.metrics import confusion_matrix, balanced_accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
import os, numpy as np, pandas as pd

req = [
    "Xtr_5","Xva_5","Xte_5",
    "ytr_5","yva_5","yte_5",
    "panel_local_idx_5","panel_genes_5",
    "le5","OUT_DIR","RANDOM_SEED"
]
missing = [k for k in req if k not in globals()]
if missing:
    raise NameError(f"Missing in memory: {missing}. Run secondary Cells 6–8 first.")

#Basics
K = len(le5.classes_)
classes_5 = list(le5.classes_)

# ---- panel matrices ----
Xtr_p_5 = Xtr_5[:, panel_local_idx_5]
Xva_p_5 = Xva_5[:, panel_local_idx_5]
Xte_p_5 = Xte_5[:, panel_local_idx_5]

def zscore_rows(X):
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True) + 1e-8
    return (X - mu) / sd

def cosine_sim(A, B):
    A2 = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-8)
    B2 = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-8)
    return A2 @ B2.T

def eval_block(y_true, y_pred, tag):
    b = float(balanced_accuracy_score(y_true, y_pred))
    f = float(f1_score(y_true, y_pred, average="macro"))
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(K))
    print(f"\n[{tag}] bACC={b:.3f} | macro-F1={f:.3f}")
    print(cm)
    return b, f, cm

# =========================================================
# A) Compact centroid baseline (on the compact panel)
# =========================================================
Xtrz_5 = zscore_rows(Xtr_p_5)
centroids_compact_5 = np.stack([Xtrz_5[ytr_5 == k].mean(axis=0) for k in range(K)], axis=0)

va_pred_c_5 = np.argmax(cosine_sim(zscore_rows(Xva_p_5), centroids_compact_5), axis=1)
te_pred_c_5 = np.argmax(cosine_sim(zscore_rows(Xte_p_5), centroids_compact_5), axis=1)

va_b_c_5, va_f_c_5, cm_va_c_5 = eval_block(yva_5, va_pred_c_5, "Compact Centroid_5/VAL")
te_b_c_5, te_f_c_5, cm_te_c_5 = eval_block(yte_5, te_pred_c_5, "Compact Centroid_5/TEST")

# =========================================================
# B) Compact LR baseline (on the compact panel)

lr_compact_5 = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    class_weight="balanced",
    max_iter=3000,
    n_jobs=1,
    random_state=int(RANDOM_SEED)
)
lr_compact_5.fit(Xtr_p_5, ytr_5)

va_pred_lr_5 = lr_compact_5.predict(Xva_p_5)
te_pred_lr_5 = lr_compact_5.predict(Xte_p_5)

va_b_lr_5, va_f_lr_5, cm_va_lr_5 = eval_block(yva_5, va_pred_lr_5, "Compact LR_5/VAL")
te_b_lr_5, te_f_lr_5, cm_te_lr_5 = eval_block(yte_5, te_pred_lr_5, "Compact LR_5/TEST")

# =========================================================
# Save outputs
# =========================================================
out_df_5 = pd.DataFrame([
    {"model": "compact_centroid_5", "val_bacc": va_b_c_5,  "val_macro_f1": va_f_c_5,  "test_bacc": te_b_c_5,  "test_macro_f1": te_f_c_5},
    {"model": "compact_logreg_5",   "val_bacc": va_b_lr_5, "val_macro_f1": va_f_lr_5, "test_bacc": te_b_lr_5, "test_macro_f1": te_f_lr_5},
])

out_fp_5 = os.path.join(OUT_DIR, "compact_panel_val_test_5.csv")
out_df_5.to_csv(out_fp_5, index=False)

np.savez(
    os.path.join(OUT_DIR, "compact_panel_confusions_5.npz"),
    classes=np.array(classes_5, dtype=str),
    cm_val_centroid=cm_va_c_5,
    cm_test_centroid=cm_te_c_5,
    cm_val_lr=cm_va_lr_5,
    cm_test_lr=cm_te_lr_5
)

print("\n✅ Saved:", out_fp_5)
print("✅ Saved:", os.path.join(OUT_DIR, "compact_panel_confusions_5.npz"))
out_df_5

In [ ]:
# Cell 10 (secondary) — Reject threshold selection on VAL only; apply fixed threshold to TEST (5-namespace)
# Zero behavioral changes: only renames + output filenames to avoid collisions.

import os, numpy as np, pandas as pd
from sklearn.metrics import balanced_accuracy_score, f1_score
from scipy.optimize import minimize

req = ["lr_compact_5","Xva_p_5","yva_5","Xte_p_5","yte_5","OUT_DIR","ensure_out_dir"]
missing = [k for k in req if k not in globals()]
if missing:
    raise NameError(f"Missing in memory: {missing}. Run secondary Cell 9 first.")

ensure_out_dir(OUT_DIR)

# -----------------------
# User controls
# -----------------------
MIN_COVERAGE_5 = 0.70
THR_START_5, THR_END_5, THR_STEP_5 = 0.50, 0.90, 0.01

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / (ez.sum(axis=1, keepdims=True) + 1e-12)

# -----------------------
# Temperature scaling on VAL (fit T to minimize NLL)
# -----------------------
logits_va_5 = lr_compact_5.decision_function(Xva_p_5)
yva_int_5 = yva_5.astype(int)

def nll_for_logT(logT):
    T = float(np.exp(logT[0]))
    p = softmax(logits_va_5 / T)
    return -float(np.mean(np.log(p[np.arange(len(yva_int_5)), yva_int_5] + 1e-12)))

res_5 = minimize(nll_for_logT, x0=np.array([0.0]), method="Nelder-Mead")
T_opt_5 = float(np.exp(res_5.x[0]))
print("Temperature T (VAL-calibrated)_5:", T_opt_5)

# -----------------------
# Build reject curve on VAL
# -----------------------
proba_va_5 = softmax(logits_va_5 / T_opt_5)
pmax_va_5  = proba_va_5.max(axis=1)
pred_va_5  = proba_va_5.argmax(axis=1)

thresholds_5 = np.round(np.arange(THR_START_5, THR_END_5 + 1e-12, THR_STEP_5), 2)
rows_5 = []

yva_arr_5 = np.asarray(yva_int_5)
pred_va_arr_5 = np.asarray(pred_va_5)

for thr in thresholds_5:
    keep = pmax_va_5 >= thr
    cov = float(keep.mean())
    n_keep = int(keep.sum())

    if n_keep == 0:
        rows_5.append({"thr": float(thr), "coverage": cov, "n_kept": 0, "bacc": np.nan, "macro_f1": np.nan})
        continue

    rows_5.append({
        "thr": float(thr),
        "coverage": cov,
        "n_kept": n_keep,
        "bacc": float(balanced_accuracy_score(yva_arr_5[keep], pred_va_arr_5[keep])),
        "macro_f1": float(f1_score(yva_arr_5[keep], pred_va_arr_5[keep], average="macro")),
    })

curve_val_5 = pd.DataFrame(rows_5)

# -----------------------
# Choose threshold using VAL ONLY
# -----------------------
eligible_5 = curve_val_5[(curve_val_5["coverage"] >= MIN_COVERAGE_5) & curve_val_5["macro_f1"].notna()].copy()

if len(eligible_5) == 0:
    chosen_5 = curve_val_5[curve_val_5["macro_f1"].notna()].copy()
    chosen_5 = chosen_5.sort_values(by=["macro_f1","coverage","thr"], ascending=[False, False, True]).head(1)
    rule_used_5 = f"FALLBACK (no thr meets coverage>= {MIN_COVERAGE_5:.2f}): max macro-F1 on VAL overall"
else:
    chosen_5 = eligible_5.sort_values(by=["macro_f1","coverage","thr"], ascending=[False, False, True]).head(1)
    rule_used_5 = f"VAL-only: max macro-F1 with coverage>= {MIN_COVERAGE_5:.2f}"

chosen_thr_5 = float(chosen_5["thr"].iloc[0])

print("\n--- Threshold selection (VAL only)_5 ---")
print("Rule:", rule_used_5)
print(f"Chosen thr={chosen_thr_5:.2f} | VAL coverage={float(chosen_5['coverage'].iloc[0]):.3f} | VAL macro-F1={float(chosen_5['macro_f1'].iloc[0]):.3f}")

# -----------------------
# Apply FIXED threshold to TEST
# -----------------------
logits_te_5 = lr_compact_5.decision_function(Xte_p_5)
proba_te_5  = softmax(logits_te_5 / T_opt_5)
pmax_te_5   = proba_te_5.max(axis=1)
pred_te_5   = proba_te_5.argmax(axis=1)

keep_te_5 = pmax_te_5 >= chosen_thr_5
cov_te_5  = float(keep_te_5.mean())
n_keep_te_5 = int(keep_te_5.sum())

if n_keep_te_5 == 0:
    test_row_5 = {"thr": chosen_thr_5, "coverage": cov_te_5, "n_kept": 0, "bacc": np.nan, "macro_f1": np.nan}
else:
    test_row_5 = {
        "thr": chosen_thr_5,
        "coverage": cov_te_5,
        "n_kept": n_keep_te_5,
        "bacc": float(balanced_accuracy_score(yte_5[keep_te_5], pred_te_5[keep_te_5])),
        "macro_f1": float(f1_score(yte_5[keep_te_5], pred_te_5[keep_te_5], average="macro")),
    }

test_applied_5 = pd.DataFrame([test_row_5])
test_applied_5["T_opt"] = T_opt_5
test_applied_5["min_coverage_rule"] = MIN_COVERAGE_5
test_applied_5["threshold_rule"] = rule_used_5

print("\n--- Apply fixed threshold to TEST_5 ---")
print(test_applied_5.to_string(index=False))

# -----------------------
# Save
# -----------------------
curve_fp_5  = os.path.join(OUT_DIR, "reject_curve_compactLR_VAL_calibrated_5.csv")
choice_fp_5 = os.path.join(OUT_DIR, "reject_choice_compactLR_from_VAL_5.csv")
test_fp_5   = os.path.join(OUT_DIR, "reject_applied_compactLR_TEST_fixed_from_VAL_5.csv")

curve_val_5.to_csv(curve_fp_5, index=False)

chosen_out_5 = chosen_5.copy()
chosen_out_5["T_opt"] = T_opt_5
chosen_out_5["min_coverage_rule"] = MIN_COVERAGE_5
chosen_out_5["threshold_rule"] = rule_used_5
chosen_out_5.to_csv(choice_fp_5, index=False)

test_applied_5.to_csv(test_fp_5, index=False)

print("\n✅ Saved:")
print(" -", curve_fp_5)
print(" -", choice_fp_5)
print(" -", test_fp_5)

In [ ]:
# Cell 11 (secondary) — 5-seed robustness (tuned selector per seed) + mean±SD


import os, time
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, balanced_accuracy_score

req = ["X_tumor", "y5", "feature_cols", "le5", "OUT_DIR", "ensure_out_dir"]
missing = [k for k in req if k not in globals()]
if missing:
    raise NameError(f"Missing in memory: {missing}. Run Cells 1–4 first.")

ensure_out_dir(OUT_DIR)

# Settings
SEEDS = [123, 124, 125, 126, 127]
V = 3000

# selector tuning grids
C_SEL_GRID = [0.1, 0.3, 0.6, 1.0, 2.0]
PANEL_GRID = [60, 80, 100, 120]

# LR tuning (VAL-only)
C_TUNE = [0.3, 1.0, 3.0]

LR_MAX_ITER  = 4000
SEL_MAX_ITER = 1500
SEL_TOL      = 1e-3

SAVE_SPLITS_PER_SEED = True
STABLE_PANEL_SIZE = 80

# Helpers
def robust_scale_train(X_train, X):
    med = np.median(X_train, axis=0)
    mad = np.median(np.abs(X_train - med), axis=0) + 1e-6
    return (X - med) / mad, med, mad

def eval_metrics(y_true, y_pred):
    return {
        "bacc": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro"))
    }

def split_70_15_15_indices(y, seed):
    all_idx = np.arange(len(y))
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=int(seed))
    tr_idx, tmp_idx = next(sss1.split(all_idx, y))

    ytmp = y[tmp_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=int(seed))
    va_rel, te_rel = next(sss2.split(tmp_idx, ytmp))

    va_idx = tmp_idx[va_rel]
    te_idx = tmp_idx[te_rel]
    return tr_idx.astype(int), va_idx.astype(int), te_idx.astype(int)

def train_only_prefilter_impute_robustscale(Xtr_raw, Xva_raw, Xte_raw, V, feature_cols):
    imp_rank = SimpleImputer(strategy="median")
    Xtr_imp_rank = imp_rank.fit_transform(Xtr_raw)

    var = np.var(Xtr_imp_rank, axis=0)
    ranked = np.argsort(var)[::-1]
    v_idx = ranked[:int(V)]

    Xtr_v = Xtr_raw[:, v_idx]
    Xva_v = Xva_raw[:, v_idx]
    Xte_v = Xte_raw[:, v_idx]

    imp = SimpleImputer(strategy="median")
    Xtr_i = imp.fit_transform(Xtr_v)
    Xva_i = imp.transform(Xva_v)
    Xte_i = imp.transform(Xte_v)

    Xtr, med, mad = robust_scale_train(Xtr_i, Xtr_i)
    Xva = (Xva_i - med) / mad
    Xte = (Xte_i - med) / mad

    Xtr = Xtr.astype(np.float32)
    Xva = Xva.astype(np.float32)
    Xte = Xte.astype(np.float32)

    selected_genes = np.array(feature_cols, dtype=str)[v_idx]
    return Xtr, Xva, Xte, v_idx, selected_genes

def fit_l1_selector_scores(X, y, C, seed):
    sel = LogisticRegression(
        penalty="l1",
        solver="saga",
        C=float(C),
        class_weight="balanced",
        max_iter=int(SEL_MAX_ITER),
        tol=float(SEL_TOL),
        n_jobs=1,
        random_state=int(seed)
    )
    sel.fit(X, y)
    return np.mean(np.abs(sel.coef_), axis=0)

def tune_lr_C_on_val(Xtr, ytr, Xva, yva, C_grid, seed):
    bestC, bestF = None, -1.0
    for C in C_grid:
        m = LogisticRegression(
            penalty="l2",
            solver="lbfgs",
            C=float(C),
            class_weight="balanced",
            max_iter=int(LR_MAX_ITER),
            n_jobs=1,
            random_state=int(seed)
        )
        m.fit(Xtr, ytr)
        f = float(f1_score(yva, m.predict(Xva), average="macro"))
        if f > bestF:
            bestF, bestC = f, float(C)
    return float(bestC), float(bestF)

def mean_sd(x):
    x = np.asarray(x, dtype=float)
    return float(np.mean(x)), float(np.std(x, ddof=0))

# Main loop
rows = []
panel_counter = Counter()
t0 = time.time()

V_eff = int(min(V, X_tumor.shape[1]))
print(f"Running 5-seed robustness_5 | V={V_eff} | seeds={SEEDS}")
print(f"Tuning selector on VAL: C_SEL={C_SEL_GRID}, PANEL={PANEL_GRID}")

for s in SEEDS:
    tr_idx, va_idx, te_idx = split_70_15_15_indices(y5, seed=s)

    if SAVE_SPLITS_PER_SEED:
        split_fp = os.path.join(OUT_DIR, f"stage2_split_seed{int(s)}_5.npz")
        np.savez(
            split_fp,
            tr_idx=tr_idx,
            va_idx=va_idx,
            te_idx=te_idx,
            classes=np.array(le5.classes_, dtype=str),
            random_seed=int(s)
        )

    Xtr_raw, ytr = X_tumor[tr_idx], y5[tr_idx]
    Xva_raw, yva = X_tumor[va_idx], y5[va_idx]
    Xte_raw, yte = X_tumor[te_idx], y5[te_idx]

    Xtr, Xva, Xte, v_idx, selected_genes = train_only_prefilter_impute_robustscale(
        Xtr_raw, Xva_raw, Xte_raw, V_eff, feature_cols
    )

    bestC_base, _ = tune_lr_C_on_val(Xtr, ytr, Xva, yva, C_TUNE, seed=s)

    lr_base_val = LogisticRegression(
        penalty="l2", solver="lbfgs", C=float(bestC_base),
        class_weight="balanced", max_iter=int(LR_MAX_ITER),
        n_jobs=1, random_state=int(s)
    )
    lr_base_val.fit(Xtr, ytr)
    met_va_base = eval_metrics(yva, lr_base_val.predict(Xva))

    lr_base_test = LogisticRegression(
        penalty="l2", solver="lbfgs", C=float(bestC_base),
        class_weight="balanced", max_iter=int(LR_MAX_ITER),
        n_jobs=1, random_state=int(s)
    )
    Xtrva = np.vstack([Xtr, Xva])
    ytrva = np.concatenate([ytr, yva])
    lr_base_test.fit(Xtrva, ytrva)
    met_te_base = eval_metrics(yte, lr_base_test.predict(Xte))

    best_panel = {"mf1": -1.0, "C_SEL": None, "PANEL_SIZE": None, "panel_idx": None}
    scores_by_C = {C_SEL: fit_l1_selector_scores(Xtr, ytr, C=float(C_SEL), seed=s) for C_SEL in C_SEL_GRID}

    for C_SEL in C_SEL_GRID:
        score = scores_by_C[C_SEL]
        rank = np.argsort(score)[::-1]
        for Psz in PANEL_GRID:
            panel_local_idx = rank[:int(Psz)]
            Xtr_p = Xtr[:, panel_local_idx]
            Xva_p = Xva[:, panel_local_idx]

            lr_eval = LogisticRegression(
                penalty="l2", solver="lbfgs",
                class_weight="balanced", max_iter=int(LR_MAX_ITER),
                n_jobs=1, random_state=int(s)
            )
            lr_eval.fit(Xtr_p, ytr)
            mf1 = float(f1_score(yva, lr_eval.predict(Xva_p), average="macro"))

            if mf1 > best_panel["mf1"]:
                best_panel.update({
                    "mf1": mf1,
                    "C_SEL": float(C_SEL),
                    "PANEL_SIZE": int(Psz),
                    "panel_idx": panel_local_idx.copy()
                })

    C_SEL_best  = best_panel["C_SEL"]
    PANEL_best  = best_panel["PANEL_SIZE"]
    panel_local_idx = best_panel["panel_idx"]

    panel_genes = np.array(selected_genes, dtype=str)[panel_local_idx]
    for g in panel_genes:
        panel_counter[str(g)] += 1

    Xtr_p = Xtr[:, panel_local_idx]
    Xva_p = Xva[:, panel_local_idx]
    Xte_p = Xte[:, panel_local_idx]

    bestC_comp, _ = tune_lr_C_on_val(Xtr_p, ytr, Xva_p, yva, C_TUNE, seed=s)

    lr_comp_val = LogisticRegression(
        penalty="l2", solver="lbfgs", C=float(bestC_comp),
        class_weight="balanced", max_iter=int(LR_MAX_ITER),
        n_jobs=1, random_state=int(s)
    )
    lr_comp_val.fit(Xtr_p, ytr)
    met_va_comp = eval_metrics(yva, lr_comp_val.predict(Xva_p))

    lr_comp_test = LogisticRegression(
        penalty="l2", solver="lbfgs", C=float(bestC_comp),
        class_weight="balanced", max_iter=int(LR_MAX_ITER),
        n_jobs=1, random_state=int(s)
    )
    Xtrva_p = np.vstack([Xtr_p, Xva_p])
    ytrva_p = np.concatenate([ytr, yva])
    lr_comp_test.fit(Xtrva_p, ytrva_p)
    met_te_comp = eval_metrics(yte, lr_comp_test.predict(Xte_p))

    rows.append({
        "seed": int(s),
        "V": int(V_eff),

        "baseline_lr_bestC": float(bestC_base),
        "baseline_lr_val_macro_f1": met_va_base["macro_f1"],
        "baseline_lr_test_macro_f1": met_te_base["macro_f1"],

        "compact_sel_bestC_SEL": float(C_SEL_best),
        "compact_sel_bestPANEL": int(PANEL_best),

        "compact_lr_bestC": float(bestC_comp),
        "compact_lr_val_macro_f1": met_va_comp["macro_f1"],
        "compact_lr_test_macro_f1": met_te_comp["macro_f1"],
    })

    print(
        f"seed={s} | baseLR TEST F1={met_te_base['macro_f1']:.3f} (C={bestC_base}) | "
        f"compact(TEST F1={met_te_comp['macro_f1']:.3f}) sel(C_SEL={C_SEL_best}, P={PANEL_best}) lrC={bestC_comp}"
    )

per_seed = pd.DataFrame(rows)

m1, s1 = mean_sd(per_seed["baseline_lr_test_macro_f1"])
m2, s2 = mean_sd(per_seed["compact_lr_test_macro_f1"])

summary = pd.DataFrame([{
    "metric": "TEST macro-F1",
    "baseline_LR(V) mean": m1,
    "baseline_LR(V) sd": s1,
    "compact_LR(panel) mean": m2,
    "compact_LR(panel) sd": s2,
    "seeds": str(SEEDS),
    "V": int(V_eff),
    "selector_grids": f"C_SEL={C_SEL_GRID}; PANEL={PANEL_GRID}"
}])

freq_df = pd.DataFrame(
    [{"GENE": g, "seed_count": c} for g, c in panel_counter.items()]
).sort_values(["seed_count", "GENE"], ascending=[False, True])

stable_panel = freq_df.head(int(STABLE_PANEL_SIZE))["GENE"].astype(str).tolist()

per_seed_fp = os.path.join(OUT_DIR, "robustness_5seeds_per_seed_5.csv")
summary_fp  = os.path.join(OUT_DIR, "robustness_5seeds_summary_5.csv")
freq_fp     = os.path.join(OUT_DIR, "descriptive_gene_inclusion_frequency_TOP80_5seeds_5.csv")
stable_fp   = os.path.join(OUT_DIR, "descriptive_TOP80_by_inclusion_frequency_5seeds_5.csv")
stable_alias_fp = os.path.join(OUT_DIR, "stable_compact_panel_TOP80_5seeds_5.csv")

per_seed.to_csv(per_seed_fp, index=False)
summary.to_csv(summary_fp, index=False)
freq_df.to_csv(freq_fp, index=False)
pd.Series(stable_panel, name="GENE").to_csv(stable_fp, index=False)
pd.Series(stable_panel, name="GENE").to_csv(stable_alias_fp, index=False)

print("\n✅ Saved:", per_seed_fp)
print("✅ Saved:", summary_fp)
print("✅ Saved:", freq_fp)
print("✅ Saved:", stable_fp)
print("✅ Saved:", stable_alias_fp)

if SAVE_SPLITS_PER_SEED:
    print("Saved per-seed Stage-2 split files: stage2_split_seed{seed}_5.npz")

print("\n=== 5-seed summary_5 (mean±SD) ===")
print(summary.to_string(index=False))

print("\nDescriptive stability (NOT a locked external signature). Top 20 by inclusion frequency:")
print(stable_panel[:20])

print(f"\nElapsed: {(time.time()-t0)/60:.2f} min")

In [ ]:
# ============================================
# Cell 12 — PRIMARY (4-class) FAIR SPLIT + persist by SAMPLE_ID
# Rules enforced:
#  - No LabelEncoder for 4-class
#  - Use lab_map with CLASS_ORDER_4
#  - Save/reapply via SAMPLE_IDs (publication-safe)
#  - Do NOT redefine df/feature_cols/X_mat/APPLY_LOG1P
# ============================================

import os, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

req = ["df", "X_mat", "y6_str", "OUT_DIR", "ensure_out_dir"]
missing = [k for k in req if k not in globals()]
if missing:
    raise NameError(f"Missing in memory: {missing}. Run Cells 1–5 first. Missing={missing}")

ensure_out_dir(OUT_DIR)

SEED_4 = 123
np.random.seed(int(SEED_4))

CLASS_ORDER_4 = ["Basal", "Her2", "LumA", "LumB"]
allowed_4 = np.array(CLASS_ORDER_4, dtype=object)

mask_4 = np.isin(np.array(y6_str, dtype=object), allowed_4)

X_4_full = X_mat[mask_4].astype(np.float32)
y_4_str = np.array(y6_str, dtype=object)[mask_4].astype(str)

# SAMPLE_IDs aligned to X_4_full rows (IMPORTANT)
sid_4 = df.loc[mask_4, "SAMPLE_ID"].astype(str).to_numpy()

# NO LabelEncoder: enforce your order explicitly
lab_map_4 = {c: i for i, c in enumerate(CLASS_ORDER_4)}
y_4 = np.array([lab_map_4[s] for s in y_4_str], dtype=int)

print("PRIMARY 4-class size:", X_4_full.shape)
print("PRIMARY counts:", pd.Series(y_4_str).value_counts().to_dict())

# 70/15/15 split on indices in the 4-class subset
all_idx_4 = np.arange(len(y_4))
sss1_4 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=int(SEED_4))
tr_idx_4, tmp_idx_4 = next(sss1_4.split(all_idx_4, y_4))

y_tmp_4 = y_4[tmp_idx_4]
sss2_4 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=int(SEED_4))
va_rel_4, te_rel_4 = next(sss2_4.split(tmp_idx_4, y_tmp_4))
va_idx_4 = tmp_idx_4[va_rel_4]
te_idx_4 = tmp_idx_4[te_rel_4]

print("PRIMARY split sizes:", "train", len(tr_idx_4), "val", len(va_idx_4), "test", len(te_idx_4))

split_fp_4 = os.path.join(OUT_DIR, f"primary4_split_seed{int(SEED_4)}.npz")
np.savez(
    split_fp_4,
    tr_sid=sid_4[tr_idx_4],
    va_sid=sid_4[va_idx_4],
    te_sid=sid_4[te_idx_4],
    class_order=np.array(CLASS_ORDER_4, dtype=str),
    seed=int(SEED_4),
)
print("✅ Saved PRIMARY split:", split_fp_4)

In [ ]:
# ============================================
# Cell 13 — PRIMARY (4-class) preprocess (NO leakage)
# Outputs:
#  - Xtr_4_raw/Xva_4_raw/Xte_4_raw (FULL gene space)
#  - ytr_4/yva_4/yte_4
#  - Xtr_4/Xva_4/Xte_4 (V=3000 processed)
#  - selected_genes_4, v_idx_4
# ============================================

import os, numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

req = ["X_4_full", "y_4", "sid_4", "feature_cols", "OUT_DIR", "ensure_out_dir"]
missing = [k for k in req if k not in globals()]
if missing:
    raise NameError(f"Missing in memory: {missing}. Run Cell 12 first. Missing={missing}")

ensure_out_dir(OUT_DIR)

SEED_4 = 123
split_fp_4 = os.path.join(OUT_DIR, f"primary4_split_seed{int(SEED_4)}.npz")
if not os.path.exists(split_fp_4):
    raise FileNotFoundError(f"Missing split file: {split_fp_4}. Run Cell 12 first.")

spl_4 = np.load(split_fp_4, allow_pickle=True)
tr_sid_4 = spl_4["tr_sid"].astype(str)
va_sid_4 = spl_4["va_sid"].astype(str)
te_sid_4 = spl_4["te_sid"].astype(str)

sid_to_idx_4 = {s: i for i, s in enumerate(sid_4)}
def map_sids_to_idx_4(sids):
    miss = [s for s in sids if s not in sid_to_idx_4]
    if miss:
        raise RuntimeError(f"Split SAMPLE_IDs not found in current 4-class subset (first 10): {miss[:10]}")
    return np.array([sid_to_idx_4[s] for s in sids], dtype=int)

tr_idx_4 = map_sids_to_idx_4(tr_sid_4)
va_idx_4 = map_sids_to_idx_4(va_sid_4)
te_idx_4 = map_sids_to_idx_4(te_sid_4)

# RAW (FULL gene space)
Xtr_4_raw = X_4_full[tr_idx_4]
Xva_4_raw = X_4_full[va_idx_4]
Xte_4_raw = X_4_full[te_idx_4]

ytr_4 = y_4[tr_idx_4]
yva_4 = y_4[va_idx_4]
yte_4 = y_4[te_idx_4]

V_4 = int(min(3000, X_4_full.shape[1]))
print("PRIMARY Using V_4 =", V_4)

# Train-only variance ranking
imp_rank_4 = SimpleImputer(strategy="median")
Xtr_4_imp = imp_rank_4.fit_transform(Xtr_4_raw)
var_4 = np.var(Xtr_4_imp, axis=0)
ranked_4 = np.argsort(var_4)[::-1]
v_idx_4 = ranked_4[:V_4]

Xtr_4_v = Xtr_4_raw[:, v_idx_4]
Xva_4_v = Xva_4_raw[:, v_idx_4]
Xte_4_v = Xte_4_raw[:, v_idx_4]

# Train-only impute + scale
imp_4 = SimpleImputer(strategy="median")
sc_4 = StandardScaler()

Xtr_4 = sc_4.fit_transform(imp_4.fit_transform(Xtr_4_v)).astype(np.float32)
Xva_4 = sc_4.transform(imp_4.transform(Xva_4_v)).astype(np.float32)
Xte_4 = sc_4.transform(imp_4.transform(Xte_4_v)).astype(np.float32)

selected_genes_4 = np.array(feature_cols, dtype=str)[v_idx_4]

print("✅ PRIMARY Xtr_4/Xva_4/Xte_4:", Xtr_4.shape, Xva_4.shape, Xte_4.shape)
print("✅ selected_genes_4 example:", selected_genes_4[:10].tolist())

In [ ]:
# ============================================
# Cell 14 — Export PAM50 centroids from genefu
# Outputs (OUT_DIR):
# - pam50_centroids.tsv (GENE + centroid columns)
# - pam50_genes.tsv (GENE list)
# - pam50_export_R.log
# ============================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, shutil, subprocess, textwrap

DATA_DIR = "/content/drive/MyDrive/EXPORT_FOR_COLAB"
if "OUT_DIR" not in globals():
    OUT_DIR = os.path.join(DATA_DIR, "ML_RESULTS_UPGRADED")
ensure_out_dir(OUT_DIR)

print("✅ OUT_DIR =", OUT_DIR)

def ensure_r():
    rscript = shutil.which("Rscript")
    if rscript:
        print("✅ Rscript found:", rscript)
        return
    print("Rscript not found. Installing R ...")
    subprocess.check_call(["apt-get", "-qq", "update"])
    subprocess.check_call(["apt-get", "-qq", "install", "-y", "r-base", "r-base-dev"])
    rscript = shutil.which("Rscript")
    if not rscript:
        raise RuntimeError("R installation failed; Rscript still not found.")
    print("✅ Installed Rscript:", rscript)

ensure_r()

r_script = textwrap.dedent(r"""
options(warn=1)
repos <- "https://cloud.r-project.org"
options(repos=c(CRAN=repos))

if (!requireNamespace("BiocManager", quietly=TRUE)) {
  install.packages("BiocManager", repos=repos)
}
if (!requireNamespace("genefu", quietly=TRUE)) {
  BiocManager::install("genefu", update=FALSE, ask=FALSE)
}

suppressPackageStartupMessages(library(genefu))

out_dir <- Sys.getenv("OUT_DIR")
if (is.null(out_dir) || out_dir == "") stop("OUT_DIR env var not set.")

data(pam50)

sbt_mod <- NULL
if (exists("pam50.robust")) sbt_mod <- pam50.robust
if (is.null(sbt_mod) && exists("pam50")) sbt_mod <- pam50
if (is.null(sbt_mod)) stop("Could not find pam50 / pam50.robust after data(pam50).")

cat("PAM50 object class:\n"); print(class(sbt_mod)); cat("\n")
cat("PAM50 object names:\n"); print(names(sbt_mod)); cat("\n")

cent <- NULL
for (nm in c("centroids","centroid","sbt.centroids","cla.centroids","subtype.centroids")) {
  if (!is.null(sbt_mod[[nm]])) {
    cent <- sbt_mod[[nm]]
    cat("Found centroids in field:", nm, "\n")
    break
  }
}
if (is.null(cent) && !is.null(sbt_mod$sbt.model) && !is.null(sbt_mod$sbt.model$centroids)) {
  cent <- sbt_mod$sbt.model$centroids
  cat("Found centroids in sbt_mod$sbt.model$centroids\n")
}
if (is.null(cent)) stop("Could not locate centroids matrix inside PAM50 object.")

cent <- as.matrix(cent)
cat("Centroids dim:\n"); print(dim(cent)); cat("\n")
cat("Centroids columns:\n"); print(colnames(cent)); cat("\n")

genes <- rownames(cent)
if (is.null(genes)) stop("Centroids have no rownames (genes).")

rownames(cent) <- toupper(trimws(genes))

cent_fp <- file.path(out_dir, "pam50_centroids.tsv")
genes_fp <- file.path(out_dir, "pam50_genes.tsv")

cent_df <- data.frame(GENE=rownames(cent), cent, check.names=FALSE, stringsAsFactors=FALSE)
write.table(cent_df, cent_fp, sep="\t", row.names=FALSE, quote=FALSE)
write.table(data.frame(GENE=rownames(cent), stringsAsFactors=FALSE), genes_fp,
            sep="\t", row.names=FALSE, quote=FALSE)

cat("\nSaved centroids:", cent_fp, "\n")
cat("Saved genes:", genes_fp, "\n\n")
cat("sessionInfo():\n")
print(sessionInfo())
""").strip() + "\n"

script_fp = os.path.join(OUT_DIR, "export_pam50_centroids.R")
with open(script_fp, "w") as f:
    f.write(r_script)

env = os.environ.copy()
env["OUT_DIR"] = OUT_DIR

log_fp = os.path.join(OUT_DIR, "pam50_export_R.log")
p = subprocess.run(
    ["Rscript", script_fp],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

with open(log_fp, "w") as f:
    f.write(p.stdout)

print("✅ R log saved:", log_fp)
print("----- R OUTPUT (last 80 lines) -----")
for ln in p.stdout.splitlines()[-80:]:
    print(ln)

if p.returncode != 0:
    raise RuntimeError(f"Rscript failed (exit {p.returncode}). Check log: {log_fp}")

cent_fp = os.path.join(OUT_DIR, "pam50_centroids.tsv")
genes_fp = os.path.join(OUT_DIR, "pam50_genes.tsv")
if not (os.path.exists(cent_fp) and os.path.exists(genes_fp)):
    raise RuntimeError("Expected outputs were not created. Check the R log for details.")

print("\n✅ Export done:")
print(" -", cent_fp)
print(" -", genes_fp)

In [ ]:
# ============================================
# Cell 15 — PRIMARY PAM50 nearest-centroid (4-class) + metrics
# Uses:
#  - PRIMARY split + y indices aligned to CLASS_ORDER_4 (0..3)
#  - Exported genefu centroids from Cell 14
# ============================================

import os, numpy as np, pandas as pd
from sklearn.metrics import balanced_accuracy_score, f1_score, recall_score
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import label_binarize

need = ["Xtr_4_raw","Xva_4_raw","Xte_4_raw","ytr_4","yva_4","yte_4","feature_cols","OUT_DIR"]
missing = [k for k in need if k not in globals()]
if missing:
    raise NameError(f"Missing in memory: {missing}. Run Cells 12–14 first. Missing={missing}")

CLASS_ORDER_4 = ["Basal", "Her2", "LumA", "LumB"]
K_4 = 4

cent_fp = os.path.join(OUT_DIR, "pam50_centroids.tsv")
if not os.path.exists(cent_fp):
    raise FileNotFoundError(f"Missing {cent_fp}. Run Cell 14 first.")

cent = pd.read_csv(cent_fp, sep="\t")
cent["GENE"] = cent["GENE"].astype(str).str.upper().str.strip()
centroids_df = cent.set_index("GENE")

# Keep only required centroid columns in our fixed order
for c in CLASS_ORDER_4:
    if c not in centroids_df.columns:
        raise ValueError(f"Centroid column missing: {c}. Available={list(centroids_df.columns)}")
centroids_df = centroids_df[CLASS_ORDER_4]

genes_full = np.array(feature_cols, dtype=str)
genes_full_U = np.char.upper(np.char.strip(genes_full))
gene_to_idx = {g:i for i,g in enumerate(genes_full_U)}

alias_map = {"CDCA1":"NUF2","KNTC2":"NDC80","ORC6L":"ORC6"}

pam_genes = centroids_df.index.to_numpy(dtype=str).tolist()
resolved = []
missing_after = []

for g in pam_genes:
    gU = str(g).upper().strip()
    if gU in gene_to_idx:
        resolved.append((gU, gU, gene_to_idx[gU]))
    else:
        ga = alias_map.get(gU, None)
        if ga is not None and ga.upper().strip() in gene_to_idx:
            gaU = ga.upper().strip()
            resolved.append((gU, gaU, gene_to_idx[gaU]))
        else:
            missing_after.append(gU)

idx_pam_4 = np.array([r[2] for r in resolved], dtype=int)
pam_labels_used_4 = np.array([r[0] for r in resolved], dtype=str)
alias_used_4 = [(a,b) for (a,b,_) in resolved if a != b]

print(f"PAM50 genes used after intersection (with aliases): {len(idx_pam_4)} / {len(pam_genes)}")
if alias_used_4:
    print("Alias mappings applied:")
    for a,b in alias_used_4:
        print(f"  {a} -> {b}")
if missing_after:
    print("Missing even after alias (first 20):", sorted(missing_after)[:20])

# X subset (50-ish genes)
Xtr_pam_4 = Xtr_4_raw[:, idx_pam_4].astype(np.float32)
Xva_pam_4 = Xva_4_raw[:, idx_pam_4].astype(np.float32)
Xte_pam_4 = Xte_4_raw[:, idx_pam_4].astype(np.float32)

# Centroids subset in the same gene order (by PAM labels used)
C_4 = centroids_df.loc[pam_labels_used_4, :].to_numpy(dtype=np.float32)  # (g,4)

def robust_scale_train(X_train, X):
    med = np.median(X_train, axis=0)
    mad = np.median(np.abs(X_train - med), axis=0) + 1e-6
    return (X - med) / mad, med, mad

Xtr_cs_4, med_pam_4, mad_pam_4 = robust_scale_train(Xtr_pam_4, Xtr_pam_4)
Xva_cs_4 = (Xva_pam_4 - med_pam_4) / mad_pam_4
Xte_cs_4 = (Xte_pam_4 - med_pam_4) / mad_pam_4

def corr_scores(X, Cmat):
    Xc = X - X.mean(axis=1, keepdims=True)
    Cc = Cmat - Cmat.mean(axis=0, keepdims=True)
    Xn = Xc / (np.linalg.norm(Xc, axis=1, keepdims=True) + 1e-8)
    Cn = Cc / (np.linalg.norm(Cc, axis=0, keepdims=True) + 1e-8)
    return Xn @ Cn

def softmax(S):
    S = S - S.max(axis=1, keepdims=True)
    ex = np.exp(S)
    return ex / (ex.sum(axis=1, keepdims=True) + 1e-8)

scores_va_4 = corr_scores(Xva_cs_4, C_4)
scores_te_4 = corr_scores(Xte_cs_4, C_4)

yhat_va_4 = scores_va_4.argmax(axis=1)
yhat_te_4 = scores_te_4.argmax(axis=1)

def eval_split(y_true, y_pred, score_mat, split_name):
    bacc = float(balanced_accuracy_score(y_true, y_pred))
    mf1  = float(f1_score(y_true, y_pred, average="macro"))

    rec = recall_score(y_true, y_pred, average=None, labels=np.arange(K_4))
    per_class_recall = {CLASS_ORDER_4[i]: float(rec[i]) for i in range(K_4)}

    y_bin = label_binarize(y_true, classes=np.arange(K_4))
    prob = softmax(score_mat)

    try:
        macro_roc = float(roc_auc_score(y_bin, prob, average="macro", multi_class="ovr"))
    except Exception:
        macro_roc = np.nan
    try:
        macro_pr  = float(average_precision_score(y_bin, prob, average="macro"))
    except Exception:
        macro_pr = np.nan

    print(f"\n[{split_name}] bACC={bacc:.3f} | macro-F1={mf1:.3f} | macro-ROC-AUC(ovr)={macro_roc:.3f} | macro-PR-AUC={macro_pr:.3f}")
    print("Per-class recall:", {k: round(per_class_recall[k], 3) for k in CLASS_ORDER_4})
    return bacc, mf1, macro_roc, macro_pr, per_class_recall

va_m_4 = eval_split(yva_4, yhat_va_4, scores_va_4, "PAM50(4-class)/VAL")
te_m_4 = eval_split(yte_4, yhat_te_4, scores_te_4, "PAM50(4-class)/TEST")

out_4 = pd.DataFrame([{
    "model": "PAM50_centroid(genefu_export)_4class_alias_fixed",
    "val_bacc": va_m_4[0], "val_macro_f1": va_m_4[1], "val_macro_roc_auc_ovr": va_m_4[2], "val_macro_pr_auc": va_m_4[3],
    "test_bacc": te_m_4[0], "test_macro_f1": te_m_4[1], "test_macro_roc_auc_ovr": te_m_4[2], "test_macro_pr_auc": te_m_4[3],
    "pam50_genes_used": int(len(idx_pam_4)),
    "class_order": ",".join(CLASS_ORDER_4),
}])

out_fp_4 = os.path.join(OUT_DIR, "pam50_eval_primary4_val_test.csv")
out_4.to_csv(out_fp_4, index=False)
print("\n✅ Saved:", out_fp_4)
out_4

In [ ]:
# ============================================
# Cell 16 — PRIMARY Compact panel vs PAM50 (same PRIMARY split)
# No re-loading of X/y. Uses:
#  - Xtr_4/Xva_4/Xte_4 from Cell 13 (V=3000 processed)
#  - selected_genes_4 from Cell 13
# ============================================

import os, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score, recall_score

need = ["Xtr_4","Xva_4","Xte_4","ytr_4","yva_4","yte_4","selected_genes_4","OUT_DIR"]
missing = [k for k in need if k not in globals()]
if missing:
    raise NameError(f"Missing in memory: {missing}. Run Cell 13 first. Missing={missing}")

CLASS_ORDER_4 = ["Basal", "Her2", "LumA", "LumB"]
K_4 = 4

# ---- Compact selector (TRAIN-only) -> TOP80 ----
PANEL_SIZE_4 = 80
C_SEL_4 = float(CONFIG.get("selector", {}).get("C_SEL", 0.6))
SEL_MAX_ITER_4 = int(CONFIG.get("selector", {}).get("max_iter", 1500))
SEL_TOL_4 = float(CONFIG.get("selector", {}).get("tol", 1e-3))

sel_4 = LogisticRegression(
    penalty="l1",
    solver="saga",
    C=C_SEL_4,
    class_weight="balanced",
    max_iter=SEL_MAX_ITER_4,
    tol=SEL_TOL_4,
    n_jobs=1,
    random_state=123,
)
sel_4.fit(Xtr_4, ytr_4)

score_4 = np.mean(np.abs(sel_4.coef_), axis=0)
rank_4 = np.argsort(score_4)[::-1]
panel_idx_4 = rank_4[:PANEL_SIZE_4]
panel_genes_4 = np.array(selected_genes_4, dtype=str)[panel_idx_4]

Xtr_p_4 = Xtr_4[:, panel_idx_4]
Xva_p_4 = Xva_4[:, panel_idx_4]
Xte_p_4 = Xte_4[:, panel_idx_4]

# ---- LR evaluation (VAL: train-only; TEST: refit train+val) ----
LR_C_4 = 1.0
LR_MAX_ITER_4 = int(CONFIG.get("classifier", {}).get("max_iter", 4000))

def metrics_4(y_true, y_pred):
    bacc = float(balanced_accuracy_score(y_true, y_pred))
    mf1  = float(f1_score(y_true, y_pred, average="macro"))
    rec  = recall_score(y_true, y_pred, average=None, labels=np.arange(K_4))
    per_class = {CLASS_ORDER_4[i]: float(rec[i]) for i in range(K_4)}
    return bacc, mf1, per_class

lr_val_4 = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    C=float(LR_C_4),
    class_weight="balanced",
    max_iter=LR_MAX_ITER_4,
    n_jobs=1,
    random_state=123,
)
lr_val_4.fit(Xtr_p_4, ytr_4)
va_pred_4 = lr_val_4.predict(Xva_p_4)
va_b_4, va_f_4, va_rec_4 = metrics_4(yva_4, va_pred_4)

lr_test_4 = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    C=float(LR_C_4),
    class_weight="balanced",
    max_iter=LR_MAX_ITER_4,
    n_jobs=1,
    random_state=123,
)
Xtrva_p_4 = np.vstack([Xtr_p_4, Xva_p_4])
ytrva_4 = np.concatenate([ytr_4, yva_4])
lr_test_4.fit(Xtrva_p_4, ytrva_4)

te_pred_4 = lr_test_4.predict(Xte_p_4)
te_b_4, te_f_4, te_rec_4 = metrics_4(yte_4, te_pred_4)

print(f"[PRIMARY Compact LR/VAL]  bACC={va_b_4:.3f} | macro-F1={va_f_4:.3f}")
print("Per-class recall:", {k: round(v,3) for k,v in va_rec_4.items()})
print(f"\n[PRIMARY Compact LR/TEST] bACC={te_b_4:.3f} | macro-F1={te_f_4:.3f}")
print("Per-class recall:", {k: round(v,3) for k,v in te_rec_4.items()})

out_compact_4 = pd.DataFrame([{
    "model": f"compact_logreg_primary4_TOP{int(PANEL_SIZE_4)}",
    "val_bacc": va_b_4,
    "val_macro_f1": va_f_4,
    "test_bacc": te_b_4,
    "test_macro_f1": te_f_4,
    "panel_size": int(PANEL_SIZE_4),
    "C_SEL": float(C_SEL_4),
    "LR_C": float(LR_C_4),
    **{f"val_recall_{k}": va_rec_4.get(k, np.nan) for k in CLASS_ORDER_4},
    **{f"test_recall_{k}": te_rec_4.get(k, np.nan) for k in CLASS_ORDER_4},
}])

out_fp_compact_4 = os.path.join(OUT_DIR, f"compact_eval_primary4_val_test_TOP{int(PANEL_SIZE_4)}.csv")
genes_fp_compact_4 = os.path.join(OUT_DIR, f"compact_panel_primary4_TOP{int(PANEL_SIZE_4)}.csv")

out_compact_4.to_csv(out_fp_compact_4, index=False)
pd.Series(panel_genes_4, name="GENE").to_csv(genes_fp_compact_4, index=False)

print("\n✅ Saved:")
print(" -", out_fp_compact_4)
print(" -", genes_fp_compact_4)
out_compact_4

In [ ]:
# ============================================
# Cell 17 — PRIMARY Compact LR (4-class) + macro ROC-AUC / PR-AUC (VAL + TEST)
# ============================================

import os, numpy as np, pandas as pd
from sklearn.metrics import balanced_accuracy_score, f1_score, recall_score
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import label_binarize

need = ["lr_val_4", "lr_test_4", "Xva_p_4", "Xte_p_4", "yva_4", "yte_4", "OUT_DIR"]
missing = [k for k in need if k not in globals()]
if missing:
    raise NameError(f"Missing in memory: {missing}. Run Cell 16 first. Missing={missing}")

CLASS_ORDER_4 = ["Basal", "Her2", "LumA", "LumB"]
K_4 = 4

def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    e = np.exp(Z)
    return e / (e.sum(axis=1, keepdims=True) + 1e-12)

def eval_lr_aucpr(model, X, y_true, split_name):
    y_pred = model.predict(X)
    bacc = float(balanced_accuracy_score(y_true, y_pred))
    mf1  = float(f1_score(y_true, y_pred, average="macro"))

    rec = recall_score(y_true, y_pred, average=None, labels=np.arange(K_4))
    per_class_recall = {CLASS_ORDER_4[i]: float(rec[i]) for i in range(K_4)}

    if hasattr(model, "decision_function"):
        logits = model.decision_function(X)
        prob = softmax(logits)
    else:
        prob = model.predict_proba(X)

    y_bin = label_binarize(y_true, classes=np.arange(K_4))

    try:
        macro_roc = float(roc_auc_score(y_bin, prob, average="macro", multi_class="ovr"))
    except Exception:
        macro_roc = np.nan

    try:
        macro_pr = float(average_precision_score(y_bin, prob, average="macro"))
    except Exception:
        macro_pr = np.nan

    print(
        f"\n[{split_name}] "
        f"bACC={bacc:.3f} | macro-F1={mf1:.3f} | macro-ROC-AUC(ovr)={macro_roc:.3f} | macro-PR-AUC={macro_pr:.3f}"
    )
    print("Per-class recall:", {k: round(per_class_recall[k], 3) for k in CLASS_ORDER_4})

    return {
        "bacc": bacc,
        "macro_f1": mf1,
        "macro_roc_auc_ovr": macro_roc,
        "macro_pr_auc": macro_pr,
        **{f"recall_{k}": per_class_recall[k] for k in CLASS_ORDER_4}
    }

val_m_4 = eval_lr_aucpr(lr_val_4,  Xva_p_4, yva_4, "PRIMARY Compact LR/VAL")
test_m_4 = eval_lr_aucpr(lr_test_4, Xte_p_4, yte_4, "PRIMARY Compact LR/TEST")

PANEL_SIZE_4 = int(Xva_p_4.shape[1])

out_aucpr_4 = pd.DataFrame([{
    "model": f"compact_logreg_primary4_TOP{PANEL_SIZE_4}",
    "val_bacc": val_m_4["bacc"],
    "val_macro_f1": val_m_4["macro_f1"],
    "val_macro_roc_auc_ovr": val_m_4["macro_roc_auc_ovr"],
    "val_macro_pr_auc": val_m_4["macro_pr_auc"],
    "test_bacc": test_m_4["bacc"],
    "test_macro_f1": test_m_4["macro_f1"],
    "test_macro_roc_auc_ovr": test_m_4["macro_roc_auc_ovr"],
    "test_macro_pr_auc": test_m_4["macro_pr_auc"],
    **{f"val_recall_{k}": val_m_4[f"recall_{k}"] for k in CLASS_ORDER_4},
    **{f"test_recall_{k}": test_m_4[f"recall_{k}"] for k in CLASS_ORDER_4},
}])

out_fp_aucpr_4 = os.path.join(OUT_DIR, f"compact_lr_primary4_aucpr_val_test_TOP{PANEL_SIZE_4}.csv")
out_aucpr_4.to_csv(out_fp_aucpr_4, index=False)
print("\n✅ Saved:", out_fp_aucpr_4)
out_aucpr_4

In [ ]:
# ============================================================
# CELL 18 — Build robustness_pam50_vs_compactLR_4class_per_seed.csv
# 4-class task: Basal/Her2/LumA/LumB
# For each seed:
#   - 70/15/15 split
#   - PAM50: nearest-centroid correlation using exported genefu centroids (50 genes)
#   - Compact: V=3000 variance prefilter (train-only) + L1 selector -> TOP80 (train-only) + LR tuned on VAL
#   - Refit compact LR on TRAIN+VAL; evaluate once on TEST
# Outputs:
#   - robustness_pam50_vs_compactLR_4class_per_seed.csv
#   - robustness_pam50_vs_compactLR_4class_summary.csv
# ============================================================

import os, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, balanced_accuracy_score, roc_auc_score, average_precision_score

DATA_DIR = "/content/drive/MyDrive/EXPORT_FOR_COLAB"
OUT_DIR  = os.path.join(DATA_DIR, "ML_RESULTS_UPGRADED")
X_FP     = os.path.join(DATA_DIR, "X_samples_x_genes.tsv.gz")
Y_FP     = os.path.join(DATA_DIR, "y_labels.tsv")
CENT_FP  = os.path.join(OUT_DIR, "pam50_centroids.tsv")

os.makedirs(OUT_DIR, exist_ok=True)

for fp in [X_FP, Y_FP, CENT_FP]:
    if not os.path.exists(fp):
        raise FileNotFoundError(f"Missing required file: {fp}")

# -----------------------
# Load aligned full data
# -----------------------
y_df = pd.read_csv(Y_FP, sep="\t")
X_df = pd.read_csv(X_FP, sep="\t", compression="gzip")
df = X_df.merge(y_df, on="SAMPLE_ID", how="inner")

feature_cols = [c for c in df.columns if c not in ("SAMPLE_ID", "y")]
X_all = df[feature_cols].to_numpy(dtype=np.float32)
y_all = df["y"].astype(str).to_numpy()

# -----------------------
# 4-class subset
# -----------------------
CLASS_ORDER = ["Basal", "Her2", "LumA", "LumB"]
lab_map = {c:i for i,c in enumerate(CLASS_ORDER)}

mask4 = np.isin(y_all, np.array(CLASS_ORDER, dtype=object))
X4_full = X_all[mask4]
y4_str  = y_all[mask4]
y4      = np.array([lab_map[s] for s in y4_str], dtype=int)

print("4-class size:", X4_full.shape, "counts:", pd.Series(y4_str).value_counts().to_dict())

# -----------------------
# PAM50 centroids (exported from genefu)
# -----------------------
cent = pd.read_csv(CENT_FP, sep="\t")
cent["GENE"] = cent["GENE"].astype(str).str.upper().str.strip()
centroids_df = cent.set_index("GENE")[CLASS_ORDER]  # 50 x 4 expected

ALIAS_MAP = {"CDCA1":"NUF2","KNTC2":"NDC80","ORC6L":"ORC6"}

genes_full = np.array(feature_cols, dtype=str)
genes_full_U = np.char.upper(np.char.strip(genes_full))
gene_to_idx = {g:i for i,g in enumerate(genes_full_U)}

pam_genes = centroids_df.index.to_list()
used_pairs, idx_pam = [], []
for g in pam_genes:
    gU = str(g).upper().strip()
    if gU in gene_to_idx:
        idx_pam.append(gene_to_idx[gU]); used_pairs.append((gU, gU))
    else:
        a = ALIAS_MAP.get(gU, None)
        if a is not None:
            aU = a.upper().strip()
            if aU in gene_to_idx:
                idx_pam.append(gene_to_idx[aU]); used_pairs.append((gU, aU))

idx_pam = np.array(idx_pam, dtype=int)
if len(idx_pam) != 50:
    raise RuntimeError(f"PAM50 overlap not 50/50 (got {len(idx_pam)}). Fix gene naming/aliases first.")

# centroids matrix in PAM gene order
C_pam = centroids_df.loc[[p[0] for p in used_pairs], :].to_numpy(dtype=np.float32)  # 50x4

def robust_center_scale_train(X_train, X):
    med = np.median(X_train, axis=0)
    mad = np.median(np.abs(X_train - med), axis=0) + 1e-6
    return (X - med) / mad

def corr_scores(X, C):
    Xc = X - X.mean(axis=1, keepdims=True)
    Cc = C - C.mean(axis=0, keepdims=True)
    Xn = Xc / (np.linalg.norm(Xc, axis=1, keepdims=True) + 1e-8)
    Cn = Cc / (np.linalg.norm(Cc, axis=0, keepdims=True) + 1e-8)
    return Xn @ Cn

def softmax(S):
    S = S - S.max(axis=1, keepdims=True)
    ex = np.exp(S)
    return ex / (ex.sum(axis=1, keepdims=True) + 1e-12)

def split_70_15_15(y, seed):
    all_idx = np.arange(len(y))
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=int(seed))
    tr_idx, tmp_idx = next(sss1.split(all_idx, y))
    ytmp = y[tmp_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=int(seed))
    va_rel, te_rel = next(sss2.split(tmp_idx, ytmp))
    va_idx = tmp_idx[va_rel]
    te_idx = tmp_idx[te_rel]
    return tr_idx.astype(int), va_idx.astype(int), te_idx.astype(int)

def metrics_from_probs(y_true, prob):
    y_pred = prob.argmax(axis=1)
    bacc = float(balanced_accuracy_score(y_true, y_pred))
    mf1  = float(f1_score(y_true, y_pred, average="macro"))
    y_bin = label_binarize(y_true, classes=np.arange(4))
    try:
        roc = float(roc_auc_score(y_bin, prob, average="macro", multi_class="ovr"))
    except Exception:
        roc = np.nan
    try:
        pr  = float(average_precision_score(y_bin, prob, average="macro"))
    except Exception:
        pr = np.nan
    return bacc, mf1, roc, pr

# -----------------------
# Robustness loop
# -----------------------
SEEDS = [123,124,125,126,127]
V = 3000
PANEL_SIZE = 80
C_SEL = 0.6
C_TUNE = [0.3, 1.0, 3.0]

rows = []

for seed in SEEDS:
    tr_idx, va_idx, te_idx = split_70_15_15(y4, seed)

    Xtr_raw, ytr = X4_full[tr_idx], y4[tr_idx]
    Xva_raw, yva = X4_full[va_idx], y4[va_idx]
    Xte_raw, yte = X4_full[te_idx], y4[te_idx]

    # -------- PAM50 (centroid) --------
    Xtr_pam = Xtr_raw[:, idx_pam]
    Xte_pam = Xte_raw[:, idx_pam]
    Xtr_cs  = robust_center_scale_train(Xtr_pam, Xtr_pam)
    Xte_cs  = robust_center_scale_train(Xtr_pam, Xte_pam)
    scores_te = corr_scores(Xte_cs, C_pam)
    prob_te_pam = softmax(scores_te)
    pam_b, pam_f, pam_roc, pam_pr = metrics_from_probs(yte, prob_te_pam)

    # -------- Compact pipeline --------
    V_eff = int(min(V, Xtr_raw.shape[1]))

    # Train-only variance rank in full space
    imp_rank = SimpleImputer(strategy="median")
    Xtr_imp = imp_rank.fit_transform(Xtr_raw)
    var = np.var(Xtr_imp, axis=0)
    ranked = np.argsort(var)[::-1]
    v_idx = ranked[:V_eff]

    Xtr_v = Xtr_raw[:, v_idx]
    Xva_v = Xva_raw[:, v_idx]
    Xte_v = Xte_raw[:, v_idx]

    # Train-only impute + scale
    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()
    Xtr = sc.fit_transform(imp.fit_transform(Xtr_v)).astype(np.float32)
    Xva = sc.transform(imp.transform(Xva_v)).astype(np.float32)
    Xte = sc.transform(imp.transform(Xte_v)).astype(np.float32)

    # Selector (train-only) -> TOP80
    sel = LogisticRegression(
        penalty="l1", solver="saga", C=float(C_SEL),
        class_weight="balanced", max_iter=1500, tol=1e-3,
        n_jobs=1, random_state=int(seed)
    )
    sel.fit(Xtr, ytr)
    score = np.mean(np.abs(sel.coef_), axis=0)
    panel_idx = np.argsort(score)[::-1][:PANEL_SIZE]

    Xtr_p = Xtr[:, panel_idx]
    Xva_p = Xva[:, panel_idx]
    Xte_p = Xte[:, panel_idx]

    # Tune LR C on VAL (macro-F1)
    bestC, bestF = None, -1.0
    for Cc in C_TUNE:
        m = LogisticRegression(
            penalty="l2", solver="lbfgs", C=float(Cc),
            class_weight="balanced", max_iter=4000,
            n_jobs=1, random_state=int(seed)
        )
        m.fit(Xtr_p, ytr)
        f = float(f1_score(yva, m.predict(Xva_p), average="macro"))
        if f > bestF:
            bestF, bestC = f, float(Cc)

    # Refit train+val, evaluate TEST
    lr = LogisticRegression(
        penalty="l2", solver="lbfgs", C=float(bestC),
        class_weight="balanced", max_iter=4000,
        n_jobs=1, random_state=int(seed)
    )
    Xtrva_p = np.vstack([Xtr_p, Xva_p])
    ytrva   = np.concatenate([ytr, yva])
    lr.fit(Xtrva_p, ytrva)

    prob_te_comp = lr.predict_proba(Xte_p)
    comp_b, comp_f, comp_roc, comp_pr = metrics_from_probs(yte, prob_te_comp)

    rows.append({
        "seed": int(seed),
        "pam50_test_bacc": pam_b,
        "pam50_test_macro_f1": pam_f,
        "pam50_test_macro_roc_auc_ovr": pam_roc,
        "pam50_test_macro_pr_auc": pam_pr,

        "compact_test_bacc": comp_b,
        "compact_test_macro_f1": comp_f,
        "compact_test_macro_roc_auc_ovr": comp_roc,
        "compact_test_macro_pr_auc": comp_pr,

        "compact_bestC": float(bestC),
        "V": int(V_eff),
        "panel_size": int(PANEL_SIZE),
        "C_SEL": float(C_SEL),
    })

    print(f"seed={seed} | PAM50 F1={pam_f:.3f} | Compact F1={comp_f:.3f} (bestC={bestC})")

per_seed = pd.DataFrame(rows)
out_fp = os.path.join(OUT_DIR, "robustness_pam50_vs_compactLR_4class_per_seed.csv")
per_seed.to_csv(out_fp, index=False)
print("\n✅ Saved:", out_fp)

# Summary
def mean_sd(x):
    x = np.asarray(x, dtype=float)
    return float(x.mean()), float(x.std(ddof=0))

summ_rows = []
for key in ["macro_f1","macro_roc_auc_ovr","macro_pr_auc","bacc"]:
    pm, ps = mean_sd(per_seed[f"pam50_test_{key}"])
    cm, cs = mean_sd(per_seed[f"compact_test_{key}"])
    summ_rows.append({
        "metric": key,
        "pam50_mean": pm, "pam50_sd": ps,
        "compact_mean": cm, "compact_sd": cs,
        "delta_mean(compact-pam50)": cm - pm
    })

summary = pd.DataFrame(summ_rows)
sum_fp = os.path.join(OUT_DIR, "robustness_pam50_vs_compactLR_4class_summary.csv")
summary.to_csv(sum_fp, index=False)
print("✅ Saved:", sum_fp)
display(summary)

In [ ]:
# ============================================================
# Cell 19 — Updated paper-ready figures (journal-style)
#   (1) Robustness paired point-range mean±SD across 5 seeds
#   (2) Macro ROC + Macro PR curves (seed=123) for PAM50 vs Compact LR
#
# Saves PNG+PDF into: OUT_DIR/FIGURES_4CLASS_PAPER/
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import label_binarize
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, precision_recall_curve, auc, f1_score

# -----------------------
# Paths + hard checks
# -----------------------
DATA_DIR = "/content/drive/MyDrive/EXPORT_FOR_COLAB"
OUT_DIR = globals().get("OUT_DIR", os.path.join(DATA_DIR, "ML_RESULTS_UPGRADED"))
os.makedirs(OUT_DIR, exist_ok=True)

FIG_DIR = os.path.join(OUT_DIR, "FIGURES_4CLASS_PAPER")
os.makedirs(FIG_DIR, exist_ok=True)

PER_SEED_FP = os.path.join(OUT_DIR, "robustness_pam50_vs_compactLR_4class_per_seed.csv")
CENT_FP     = os.path.join(OUT_DIR, "pam50_centroids.tsv")

X_FP = os.path.join(DATA_DIR, "X_samples_x_genes.tsv.gz")
Y_FP = os.path.join(DATA_DIR, "y_labels.tsv")

for fp in [PER_SEED_FP, CENT_FP, X_FP, Y_FP]:
    if not os.path.exists(fp):
        raise FileNotFoundError(f"Missing: {fp}")

def savefig(fig, name):
    """Save high-res PNG+PDF with tight layout."""
    png = os.path.join(FIG_DIR, f"{name}.png")
    pdf = os.path.join(FIG_DIR, f"{name}.pdf")
    fig.tight_layout()
    fig.savefig(png, bbox_inches="tight", facecolor="white", dpi=700)
    fig.savefig(pdf, bbox_inches="tight", facecolor="white", dpi=700)
    print("Saved:", png)
    print("Saved:", pdf)

def mean_sd(x):
    x = np.asarray(x, dtype=float)
    return float(x.mean()), float(x.std(ddof=0))

def pick_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of these columns were found: {candidates}")

# -----------------------
# Global style (journal-friendly)
# -----------------------
plt.rcParams.update({
    "figure.dpi": 180,
    "savefig.dpi": 700,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10.5,
    "legend.fontsize": 9.5,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans", "Liberation Sans"],
})

PAM50_COLOR   = "#1A1A1A"   # near-black
COMPACT_COLOR = "#0B7285"   # deep teal
GRID_COLOR    = "#E3E3E3"
PAIR_COLOR    = "#BDBDBD"
TEXT_COLOR    = "#222222"
DIAG_COLOR    = "#9A9A9A"

# ============================================================
# FIGURE 1 — Journal-style robustness figure (TEST)
# Paired 5-seed comparison: PAM50 vs Compact LR
# ============================================================
per_seed = pd.read_csv(PER_SEED_FP)

# Be robust to either balanced_acc or bacc naming
pam50_bacc_col   = pick_existing_col(per_seed, ["pam50_test_balanced_acc", "pam50_test_bacc"])
compact_bacc_col = pick_existing_col(per_seed, ["compact_test_balanced_acc", "compact_test_bacc"])

metric_specs = [
    ("macro_f1", "Macro-F1", "pam50_test_macro_f1", "compact_test_macro_f1"),
    ("balanced_acc", "Balanced accuracy", pam50_bacc_col, compact_bacc_col),
    ("macro_roc_auc_ovr", "ROC-AUC", "pam50_test_macro_roc_auc_ovr", "compact_test_macro_roc_auc_ovr"),
    ("macro_pr_auc", "PR-AUC", "pam50_test_macro_pr_auc", "compact_test_macro_pr_auc"),
]

rows = []
for key, lab, p_col, c_col in metric_specs:
    p = per_seed[p_col].to_numpy()
    c = per_seed[c_col].to_numpy()
    pm, ps = mean_sd(p)
    cm, cs = mean_sd(c)
    dm, ds = mean_sd(c - p)
    rows.append([key, lab, p_col, c_col, pm, ps, cm, cs, dm, ds])

plot_df = pd.DataFrame(
    rows,
    columns=[
        "key", "metric", "pam50_col", "compact_col",
        "pam50_mean", "pam50_sd", "compact_mean", "compact_sd", "delta_mean", "delta_sd"
    ]
)

x = np.arange(len(plot_df))
dx = 0.10

fig, ax = plt.subplots(figsize=(8.4, 3.8))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# paired per-seed connectors
for i in range(len(plot_df)):
    p = per_seed[plot_df.loc[i, "pam50_col"]].to_numpy()
    c = per_seed[plot_df.loc[i, "compact_col"]].to_numpy()
    for j in range(len(p)):
        ax.plot(
            [i - dx, i + dx], [p[j], c[j]],
            lw=0.9, color=PAIR_COLOR, alpha=0.85, zorder=1
        )

# mean ± SD points
ax.errorbar(
    x - dx,
    plot_df["pam50_mean"],
    yerr=plot_df["pam50_sd"],
    fmt="o",
    markersize=5.5,
    lw=1.2,
    capsize=3,
    color=PAM50_COLOR,
    label="PAM50 centroid",
    zorder=3
)

ax.errorbar(
    x + dx,
    plot_df["compact_mean"],
    yerr=plot_df["compact_sd"],
    fmt="o",
    markersize=5.5,
    lw=1.2,
    capsize=3,
    color=COMPACT_COLOR,
    label="Compact LR (TOP80)",
    zorder=3
)

# clean delta annotations
for i in range(len(plot_df)):
    top = max(
        plot_df.loc[i, "pam50_mean"] + plot_df.loc[i, "pam50_sd"],
        plot_df.loc[i, "compact_mean"] + plot_df.loc[i, "compact_sd"]
    )
    delta_val = float(plot_df.loc[i, "delta_mean"])

    if abs(delta_val) < 0.01:
        txt = f"Δ = {delta_val:+.4f}"
    else:
        txt = f"Δ = {delta_val:+.3f}"

    ax.text(
        i, top + 0.010, txt,
        ha="center", va="bottom",
        fontsize=9.2, color=TEXT_COLOR
    )

ax.set_xticks(x)
ax.set_xticklabels(plot_df["metric"], rotation=0, ha="center", color=TEXT_COLOR)
ax.set_ylabel("Test score", color=TEXT_COLOR)
ax.set_title("")

ymax = max(
    (plot_df["pam50_mean"] + plot_df["pam50_sd"]).max(),
    (plot_df["compact_mean"] + plot_df["compact_sd"]).max()
) + 0.04

ymin = min(
    (plot_df["pam50_mean"] - plot_df["pam50_sd"]).min(),
    (plot_df["compact_mean"] - plot_df["compact_sd"]).min()
) - 0.02

ax.set_ylim(max(0.65, ymin), min(1.0, ymax))
ax.grid(True, axis="y", lw=0.6, color=GRID_COLOR)
ax.legend(loc="lower right", frameon=False)

savefig(fig, "FIG1_robustness_paired_point_range_meanSD_TEST_JOURNAL")
plt.show()

# ============================================================
# FIGURE 2 — Macro ROC + Macro PR (seed=123): recompute probs cleanly
# ============================================================
SEED = 123
CLASS_ORDER = ["Basal", "Her2", "LumA", "LumB"]
lab_map = {c: i for i, c in enumerate(CLASS_ORDER)}
K = 4

# Load aligned X and y
y_df = pd.read_csv(Y_FP, sep="\t")
if not {"SAMPLE_ID", "y"}.issubset(y_df.columns):
    raise ValueError(f"y_labels.tsv must contain SAMPLE_ID and y. Found: {list(y_df.columns)}")

X_df = pd.read_csv(X_FP, sep="\t", compression="gzip")
if "SAMPLE_ID" not in X_df.columns:
    raise ValueError("X_samples_x_genes.tsv.gz must contain SAMPLE_ID")

df = X_df.merge(y_df, on="SAMPLE_ID", how="inner")
feature_cols = [c for c in df.columns if c not in ("SAMPLE_ID", "y")]
X_mat = df[feature_cols].to_numpy(dtype=np.float32)
y_str = df["y"].astype(str).to_numpy()

# Subset to 4-class
mask4 = np.isin(y_str, np.array(CLASS_ORDER, dtype=object))
X4 = X_mat[mask4]
y4_str = y_str[mask4]
y4 = np.array([lab_map[s] for s in y4_str], dtype=int)

# 70/15/15 split
all_idx = np.arange(len(y4))
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
tr_idx, tmp_idx = next(sss1.split(all_idx, y4))
ytmp = y4[tmp_idx]

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
va_rel, te_rel = next(sss2.split(tmp_idx, ytmp))
va_idx = tmp_idx[va_rel]
te_idx = tmp_idx[te_rel]

Xtr_raw, ytr = X4[tr_idx], y4[tr_idx]
Xva_raw, yva = X4[va_idx], y4[va_idx]
Xte_raw, yte = X4[te_idx], y4[te_idx]

# ---- Load centroids ----
cent = pd.read_csv(CENT_FP, sep="\t")
cent["GENE"] = cent["GENE"].astype(str).str.upper().str.strip()
centroids_df = cent.set_index("GENE")[CLASS_ORDER]  # 50 x 4

ALIAS_MAP = {"CDCA1": "NUF2", "KNTC2": "NDC80", "ORC6L": "ORC6"}

genes_full = np.array(feature_cols, dtype=str)
genes_full_U = np.char.upper(np.char.strip(genes_full))
gene_to_idx = {g: i for i, g in enumerate(genes_full_U)}

pam_genes = centroids_df.index.to_list()
used_pairs, idx_pam = [], []
for g in pam_genes:
    gU = str(g).upper().strip()
    if gU in gene_to_idx:
        idx_pam.append(gene_to_idx[gU])
        used_pairs.append((gU, gU))
    else:
        a = ALIAS_MAP.get(gU, None)
        if a is not None:
            aU = a.upper().strip()
            if aU in gene_to_idx:
                idx_pam.append(gene_to_idx[aU])
                used_pairs.append((gU, aU))

idx_pam = np.array(idx_pam, dtype=int)
if len(idx_pam) != 50:
    raise RuntimeError(f"PAM50 overlap not 50/50 (got {len(idx_pam)}). Check gene naming/aliases.")

def robust_center_scale_train(X_train, X):
    med = np.median(X_train, axis=0)
    mad = np.median(np.abs(X_train - med), axis=0) + 1e-6
    return (X - med) / mad

def corr_scores(X, C):
    Xc = X - X.mean(axis=1, keepdims=True)
    Cc = C - C.mean(axis=0, keepdims=True)
    Xn = Xc / (np.linalg.norm(Xc, axis=1, keepdims=True) + 1e-8)
    Cn = Cc / (np.linalg.norm(Cc, axis=0, keepdims=True) + 1e-8)
    return Xn @ Cn

def softmax(S):
    S = S - S.max(axis=1, keepdims=True)
    ex = np.exp(S)
    return ex / (ex.sum(axis=1, keepdims=True) + 1e-12)

# PAM50 probs on TEST
Xtr_pam = Xtr_raw[:, idx_pam]
Xte_pam = Xte_raw[:, idx_pam]
C = centroids_df.loc[[p[0] for p in used_pairs], :].to_numpy(dtype=np.float32)

Xtr_cs = robust_center_scale_train(Xtr_pam, Xtr_pam)
Xte_cs = robust_center_scale_train(Xtr_pam, Xte_pam)
scores_te_pam = corr_scores(Xte_cs, C)
proba_te_pam = softmax(scores_te_pam)

# ---- Compact LR probs on TEST (seed=123) ----
V = int(min(3000, Xtr_raw.shape[1]))

imp_rank = SimpleImputer(strategy="median")
Xtr_imp = imp_rank.fit_transform(Xtr_raw)
var = np.var(Xtr_imp, axis=0)
ranked = np.argsort(var)[::-1]
v_idx = ranked[:V]

Xtr_v = Xtr_raw[:, v_idx]
Xva_v = Xva_raw[:, v_idx]
Xte_v = Xte_raw[:, v_idx]

imp = SimpleImputer(strategy="median")
sc = StandardScaler()
Xtr = sc.fit_transform(imp.fit_transform(Xtr_v)).astype(np.float32)
Xva = sc.transform(imp.transform(Xva_v)).astype(np.float32)
Xte = sc.transform(imp.transform(Xte_v)).astype(np.float32)

# Selector (train-only)
PANEL_SIZE = 80
C_SEL = 0.6
sel = LogisticRegression(
    penalty="l1",
    solver="saga",
    C=float(C_SEL),
    class_weight="balanced",
    max_iter=1500,
    tol=1e-3,
    n_jobs=1,
    random_state=SEED
)
sel.fit(Xtr, ytr)

score = np.mean(np.abs(sel.coef_), axis=0)
panel_local_idx = np.argsort(score)[::-1][:PANEL_SIZE]

Xtr_p = Xtr[:, panel_local_idx]
Xva_p = Xva[:, panel_local_idx]
Xte_p = Xte[:, panel_local_idx]

# Tune LR C on VAL using macro-F1
C_TUNE = [0.3, 1.0, 3.0]
bestC, bestF = None, -1.0
for Cc in C_TUNE:
    m = LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        C=float(Cc),
        class_weight="balanced",
        max_iter=4000,
        n_jobs=1,
        random_state=SEED
    )
    m.fit(Xtr_p, ytr)
    f = float(f1_score(yva, m.predict(Xva_p), average="macro"))
    if f > bestF:
        bestF, bestC = f, float(Cc)

# Refit train+val -> test probabilities
lr = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    C=float(bestC),
    class_weight="balanced",
    max_iter=4000,
    n_jobs=1,
    random_state=SEED
)
Xtrva_p = np.vstack([Xtr_p, Xva_p])
ytrva = np.concatenate([ytr, yva])
lr.fit(Xtrva_p, ytrva)

proba_te_comp = lr.predict_proba(Xte_p)

# ---- Macro ROC/PR curves: interpolate mean across classes ----
y_bin = label_binarize(yte, classes=np.arange(K))

def macro_roc_curve(y_bin, prob, grid_n=600):
    grid = np.linspace(0, 1, grid_n)
    tprs = []
    for k in range(K):
        fpr, tpr, _ = roc_curve(y_bin[:, k], prob[:, k])
        tpr_i = np.interp(grid, fpr, tpr)
        tpr_i[0] = 0.0
        tprs.append(tpr_i)
    mean_tpr = np.mean(np.vstack(tprs), axis=0)
    mean_tpr[-1] = 1.0
    return grid, mean_tpr, auc(grid, mean_tpr)

def macro_pr_curve(y_bin, prob, grid_n=600):
    grid = np.linspace(0, 1, grid_n)
    precs = []
    for k in range(K):
        prec, rec, _ = precision_recall_curve(y_bin[:, k], prob[:, k])
        order = np.argsort(rec)
        rec_s = rec[order]
        prec_s = prec[order]
        prec_i = np.interp(grid, rec_s, prec_s)
        precs.append(prec_i)
    mean_prec = np.mean(np.vstack(precs), axis=0)
    return grid, mean_prec, auc(grid, mean_prec)

fpr_p, tpr_p, auc_p = macro_roc_curve(y_bin, proba_te_pam)
fpr_c, tpr_c, auc_c = macro_roc_curve(y_bin, proba_te_comp)

rec_p, prec_p, prauc_p = macro_pr_curve(y_bin, proba_te_pam)
rec_c, prec_c, prauc_c = macro_pr_curve(y_bin, proba_te_comp)

# ---- 2-panel plot ----
fig = plt.figure(figsize=(9.4, 4.1))
fig.patch.set_facecolor("white")

ax1 = fig.add_subplot(1, 2, 1)
ax1.set_facecolor("white")
ax1.plot(fpr_p, tpr_p, lw=2.0, color=PAM50_COLOR, label=f"PAM50 (macro AUC={auc_p:.3f})")
ax1.plot(fpr_c, tpr_c, lw=2.0, color=COMPACT_COLOR, label=f"Compact LR (macro AUC={auc_c:.3f})")
ax1.plot([0, 1], [0, 1], lw=1.0, color=DIAG_COLOR, ls=":")
ax1.set_xlabel("False positive rate", color=TEXT_COLOR)
ax1.set_ylabel("True positive rate", color=TEXT_COLOR)
ax1.set_title("ROC (macro OvR) — TEST (seed 123)", color=TEXT_COLOR)
ax1.grid(True, lw=0.6, color=GRID_COLOR, alpha=0.8)
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)

ax2 = fig.add_subplot(1, 2, 2)
ax2.set_facecolor("white")
ax2.plot(rec_p, prec_p, lw=2.0, color=PAM50_COLOR, label=f"PAM50 (macro AUC={prauc_p:.3f})")
ax2.plot(rec_c, prec_c, lw=2.0, color=COMPACT_COLOR, label=f"Compact LR (macro AUC={prauc_c:.3f})")
ax2.set_xlabel("Recall", color=TEXT_COLOR)
ax2.set_ylabel("Precision", color=TEXT_COLOR)
ax2.set_title("PR (macro) — TEST (seed 123)", color=TEXT_COLOR)
ax2.grid(True, lw=0.6, color=GRID_COLOR, alpha=0.8)
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)

handles, labels = ax2.get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="center left",
    bbox_to_anchor=(1.01, 0.5),
    frameon=False
)

savefig(fig, "FIG2_macro_ROC_PR_seed123_clean_2panel")
plt.show()

print("Done. Figures saved in:", FIG_DIR)

In [ ]:
# ============================================================
# Cell 20 — Clean gene lists and compute overlap (robust)
# FIXED:
#   - Handles your actual filenames with suffix "_5" (e.g., *_5seeds_5.csv)
#   - Robustly finds the stable TOP80 file via explicit candidates + glob fallback
#   - Reads TSV/CSV safely (auto-sep)
#   - Cleans gene symbols consistently
# ============================================================

import os, re, glob
import pandas as pd

OUT_DIR = "/content/drive/MyDrive/EXPORT_FOR_COLAB/ML_RESULTS_UPGRADED"

def smart_read_table(fp):
    # Auto-detect delimiter across .tsv/.csv
    return pd.read_csv(fp, sep=None, engine="python")

def clean_gene_series(s: pd.Series):
    g = s.astype(str).str.upper().str.strip()

    # remove null-ish tokens
    g = g.replace({"NAN":"", "NA":"", "N/A":"", "NONE":"", "NULL":"", "INF":"", "-INF":""})
    g = g[g != ""]

    # keep plausible symbols only
    g = g[g.str.len() > 1]
    g = g[g.str.match(r"^[A-Z0-9\-]+$")]

    # optional: drop a few common junk tokens if they appear
    junk = {"C", "M", "TM", "UB"}
    g = g[~g.isin(junk)]

    return sorted(set(g.tolist()))

def read_gene_list(fp, preferred_col="GENE"):
    df = smart_read_table(fp)
    col = preferred_col if preferred_col in df.columns else df.columns[0]
    genes = clean_gene_series(df[col])
    return genes, df, col

def pick_stable_top80_file(out_dir: str):
    """
    Priority order:
      1) stable_compact_panel... (your canonical stable list)
      2) descriptive_TOP80_by_inclusion_frequency...
      3) descriptive_gene_inclusion_frequency... (contains more than 80 rows, but still usable)
    Supports both old and new suffix patterns (with/without _5).
    """
    explicit_candidates = [
        # Older names (no _5)
        os.path.join(out_dir, "stable_compact_panel_TOP80_5seeds.csv"),
        os.path.join(out_dir, "descriptive_TOP80_by_inclusion_frequency_5seeds.csv"),
        os.path.join(out_dir, "descriptive_gene_inclusion_frequency_TOP80_5seeds.csv"),

        # Your current names (with _5)
        os.path.join(out_dir, "stable_compact_panel_TOP80_5seeds_5.csv"),
        os.path.join(out_dir, "descriptive_TOP80_by_inclusion_frequency_5seeds_5.csv"),
        os.path.join(out_dir, "descriptive_gene_inclusion_frequency_TOP80_5seeds_5.csv"),
    ]

    for fp in explicit_candidates:
        if os.path.exists(fp):
            return fp

    # Fallback: search anything TOP80-ish, then rank by preference keywords
    hits = sorted(glob.glob(os.path.join(out_dir, "*TOP80*.csv"))) + sorted(glob.glob(os.path.join(out_dir, "*TOP80*.tsv")))
    if not hits:
        hits2 = sorted(glob.glob(os.path.join(out_dir, "*TOP80*")))
        raise FileNotFoundError(
            "No TOP80-like file found in OUT_DIR.\n"
            f"OUT_DIR={out_dir}\n"
            "Found:\n" + "\n".join(hits2[:80])
        )

    def score(fp):
        name = os.path.basename(fp).lower()
        s = 0
        if "stable_compact_panel" in name: s += 100
        if "descriptive_top80_by_inclusion_frequency" in name: s += 60
        if "descriptive_gene_inclusion_frequency" in name: s += 30
        if "5seeds" in name: s += 10
        # prefer files (csv/tsv) not weird
        if name.endswith(".csv") or name.endswith(".tsv"): s += 5
        return s

    hits = sorted(hits, key=lambda x: score(x), reverse=True)
    return hits[0]

# ---- locate stable TOP80 file robustly ----
my80_fp = pick_stable_top80_file(OUT_DIR)

# ---- PAM50 genes file ----
pam_fp = os.path.join(OUT_DIR, "pam50_genes.tsv")
if not os.path.exists(pam_fp):
    raise FileNotFoundError(
        f"Missing PAM50 genes file: {pam_fp}\n"
        "Run your PAM50 export cell (the genefu export) first."
    )

# ---- load + clean ----
pam, pam_df, pam_col = read_gene_list(pam_fp, preferred_col="GENE")
my80, my80_df, my80_col = read_gene_list(my80_fp, preferred_col="GENE")

print("PAM50 file:", pam_fp, "| raw rows:", pam_df.shape[0], "| clean unique:", len(pam))
print("TOP80 file:", my80_fp, "| raw rows:", my80_df.shape[0], "| clean unique:", len(my80))

if len(pam) < 45:
    print("⚠️ WARNING: PAM50 clean gene count is <45.")
    print("   This suggests pam50_genes.tsv may not be the canonical 50 genes; check pam50_export_R.log.")

# ---- overlap ----
overlap = sorted(set(pam) & set(my80))
print("\nCLEAN overlap count:", len(overlap))
print(overlap)

In [ ]:
# ============================================================
# Cell 21 — Simple overlap check (ROBUST file names)
# Fixes:
#   - Handles your real filenames with suffix "_5" (e.g., *_5seeds_5.csv)
#   - Falls back to best TOP80-like file if explicit candidates not found
# Outputs (OUT_DIR):
#   - TOP80_overlap_with_PAM50.csv
#   - TOP80_nonoverlap_vs_PAM50.csv
# ============================================================

import os, glob
import pandas as pd

OUT_DIR = "/content/drive/MyDrive/EXPORT_FOR_COLAB/ML_RESULTS_UPGRADED"

def smart_read(fp):
    # auto-detect delimiter across csv/tsv
    return pd.read_csv(fp, sep=None, engine="python")

def clean_genes(series):
    g = series.astype(str).str.upper().str.strip()
    g = g.replace({"NAN":"", "NA":"", "N/A":"", "NONE":"", "NULL":"", "INF":"", "-INF":""})
    g = g[g != ""]
    # keep plausible HGNC-like tokens (letters/digits/hyphen)
    g = g[g.str.match(r"^[A-Z0-9\-]+$")]
    g = g[g.str.len() > 1]
    return sorted(set(g.tolist()))

def pick_stable_top80(out_dir):
    # Prefer truly "stable" list first, then descriptive lists
    explicit = [
        os.path.join(out_dir, "stable_compact_panel_TOP80_5seeds_5.csv"),
        os.path.join(out_dir, "stable_compact_panel_TOP80_5seeds.csv"),
        os.path.join(out_dir, "descriptive_TOP80_by_inclusion_frequency_5seeds_5.csv"),
        os.path.join(out_dir, "descriptive_TOP80_by_inclusion_frequency_5seeds.csv"),
        os.path.join(out_dir, "descriptive_gene_inclusion_frequency_TOP80_5seeds_5.csv"),
        os.path.join(out_dir, "descriptive_gene_inclusion_frequency_TOP80_5seeds.csv"),
    ]
    for fp in explicit:
        if os.path.exists(fp):
            return fp

    # Fallback: search any TOP80-ish file and rank by preference
    hits = sorted(glob.glob(os.path.join(out_dir, "*TOP80*.csv"))) + \
           sorted(glob.glob(os.path.join(out_dir, "*TOP80*.tsv")))
    if not hits:
        hits2 = sorted(glob.glob(os.path.join(out_dir, "*TOP80*")))
        raise FileNotFoundError("No TOP80-like file found. Found:\n" + "\n".join(hits2[:80]))

    def score(fp):
        n = os.path.basename(fp).lower()
        s = 0
        if "stable_compact_panel" in n: s += 100
        if "descriptive_top80_by_inclusion_frequency" in n: s += 60
        if "descriptive_gene_inclusion_frequency" in n: s += 30
        if "5seeds" in n: s += 10
        if n.endswith(".csv") or n.endswith(".tsv"): s += 5
        return s

    return sorted(hits, key=score, reverse=True)[0]

# ---- locate files ----
MY80_FP = pick_stable_top80(OUT_DIR)
PAM_FP  = os.path.join(OUT_DIR, "pam50_genes.tsv")

if not os.path.exists(PAM_FP):
    raise FileNotFoundError(f"Missing PAM50 genes file: {PAM_FP}\nRun your PAM50 export cell first.")

# ---- load + clean ----
my80_df = smart_read(MY80_FP)
pam_df  = pd.read_csv(PAM_FP, sep="\t")

my80_col = "GENE" if "GENE" in my80_df.columns else my80_df.columns[0]
pam_col  = "GENE" if "GENE" in pam_df.columns else pam_df.columns[0]

my80 = clean_genes(my80_df[my80_col])
pam  = clean_genes(pam_df[pam_col])

# ---- overlap ----
overlap = sorted(set(my80) & set(pam))
non_overlap = sorted(set(my80) - set(pam))

print("✅ Using stable TOP80 file:", MY80_FP)
print("   raw rows:", my80_df.shape[0], "| clean unique:", len(my80))
print("✅ Using PAM50 file:", PAM_FP)
print("   raw rows:", pam_df.shape[0], "| clean unique:", len(pam))

print("\nOverlap count:", len(overlap))
print(overlap)

# ---- export lists (useful for later biology/enrichment text) ----
ov_fp  = os.path.join(OUT_DIR, "TOP80_overlap_with_PAM50.csv")
non_fp = os.path.join(OUT_DIR, "TOP80_nonoverlap_vs_PAM50.csv")

pd.Series(overlap, name="GENE").to_csv(ov_fp, index=False)
pd.Series(non_overlap, name="GENE").to_csv(non_fp, index=False)

print("\n✅ Saved:")
print(" -", ov_fp)
print(" -", non_fp)

In [ ]:
# ============================================================
# Cell 22 — Load + clean gene lists, export overlap vs non-overlap
# FIXED:
#   - Handles BOTH filename schemes: with/without suffix "_5"
#   - Prefers stable_compact_panel first, then descriptive TOP80 lists
#   - Robust glob fallback ranking
#   - Reads TSV/CSV safely (auto delimiter)
# Outputs (OUT_DIR):
#   - TOP80_overlap_with_PAM50.csv
#   - TOP80_nonoverlap_vs_PAM50.csv
# ============================================================

import os, glob
import pandas as pd

OUT_DIR = "/content/drive/MyDrive/EXPORT_FOR_COLAB/ML_RESULTS_UPGRADED"

def smart_read_table(fp):
    # Auto-detect delimiter across .tsv/.csv
    return pd.read_csv(fp, sep=None, engine="python")

def clean_genes(iterable):
    s = pd.Series(list(iterable)).astype(str).str.upper().str.strip()
    s = s.replace({"NAN":"", "NA":"", "N/A":"", "NONE":"", "NULL":"", "INF":"", "-INF":""})
    s = s[s != ""]
    s = s[s.str.match(r"^[A-Z0-9\-]+$")]
    s = s[s.str.len() > 1]
    return sorted(set(s.tolist()))

def pick_stable_top80_file(out_dir: str):
    """
    Priority order:
      1) stable_compact_panel... (canonical stable list)
      2) descriptive_TOP80_by_inclusion_frequency...
      3) descriptive_gene_inclusion_frequency... (contains >80 rows, still usable)
    Supports both old and new suffix patterns (with/without _5).
    """
    explicit = [
        # Newer names (with _5)
        os.path.join(out_dir, "stable_compact_panel_TOP80_5seeds_5.csv"),
        os.path.join(out_dir, "descriptive_TOP80_by_inclusion_frequency_5seeds_5.csv"),
        os.path.join(out_dir, "descriptive_gene_inclusion_frequency_TOP80_5seeds_5.csv"),

        # Older names (no _5)
        os.path.join(out_dir, "stable_compact_panel_TOP80_5seeds.csv"),
        os.path.join(out_dir, "descriptive_TOP80_by_inclusion_frequency_5seeds.csv"),
        os.path.join(out_dir, "descriptive_gene_inclusion_frequency_TOP80_5seeds.csv"),
    ]
    for fp in explicit:
        if os.path.exists(fp):
            return fp

    # Fallback: any TOP80-ish file; rank by preference keywords
    hits = sorted(glob.glob(os.path.join(out_dir, "*TOP80*.csv"))) + \
           sorted(glob.glob(os.path.join(out_dir, "*TOP80*.tsv")))
    if not hits:
        hits2 = sorted(glob.glob(os.path.join(out_dir, "*TOP80*")))
        raise FileNotFoundError(
            "No TOP80-like file found in OUT_DIR.\n"
            f"OUT_DIR={out_dir}\n"
            "Found:\n" + "\n".join(hits2[:80])
        )

    def score(fp):
        name = os.path.basename(fp).lower()
        s = 0
        if "stable_compact_panel" in name: s += 100
        if "descriptive_top80_by_inclusion_frequency" in name: s += 60
        if "descriptive_gene_inclusion_frequency" in name: s += 30
        if "5seeds" in name: s += 10
        if name.endswith(".csv") or name.endswith(".tsv"): s += 5
        return s

    return sorted(hits, key=score, reverse=True)[0]

# ---- locate stable TOP80 file robustly ----
MY80_FP = pick_stable_top80_file(OUT_DIR)

# ---- PAM50 genes file ----
PAM_FP = os.path.join(OUT_DIR, "pam50_genes.tsv")
if not os.path.exists(PAM_FP):
    raise FileNotFoundError(
        f"Missing PAM50 genes file: {PAM_FP}\n"
        "Run your PAM50 export cell (genefu export) first."
    )

# ---- load + clean ----
my80_df = smart_read_table(MY80_FP)
pam_df  = pd.read_csv(PAM_FP, sep="\t")

my80_col = "GENE" if "GENE" in my80_df.columns else my80_df.columns[0]
pam_col  = "GENE" if "GENE" in pam_df.columns else pam_df.columns[0]

my80 = clean_genes(my80_df[my80_col])
pam  = clean_genes(pam_df[pam_col])

overlap = sorted(set(my80) & set(pam))
non_overlap = sorted(set(my80) - set(pam))

print("✅ Using TOP80 file:", MY80_FP)
print("   TOP80 raw rows:", my80_df.shape[0], "| clean unique:", len(my80))
print("✅ Using PAM50 file:", PAM_FP)
print("   PAM50 raw rows:", pam_df.shape[0], "| clean unique:", len(pam))
print("\nOverlap count:", len(overlap))
print("Non-overlap count:", len(non_overlap))

ov_fp  = os.path.join(OUT_DIR, "TOP80_overlap_with_PAM50.csv")
non_fp = os.path.join(OUT_DIR, "TOP80_nonoverlap_vs_PAM50.csv")
pd.Series(overlap, name="GENE").to_csv(ov_fp, index=False)
pd.Series(non_overlap, name="GENE").to_csv(non_fp, index=False)

print("\n✅ Saved:")
print(" -", ov_fp)
print(" -", non_fp)

In [ ]:
# ============================================================
# Cell 23 — Integrated biology (ROBUST, FIXED)
#   - TOP80 vs PAM50 overlap/non-overlap
#   - Enrichment (non-overlap) via Enrichr (gseapy)
#   - TOP80 multinomial LR coefficients (seed=123 split) + top genes/class
# Outputs (OUT_DIR):
#   - TOP80_overlap_with_PAM50.csv
#   - TOP80_nonoverlap_vs_PAM50.csv
#   - enrichment_TOP80_nonoverlap_enrichr.csv
#   - TOP80_multinomialLR_coefficients_4class.csv
#   - TOP80_top15_genes_per_class_4class.csv
# ============================================================

import os, glob
import numpy as np
import pandas as pd

OUT_DIR  = "/content/drive/MyDrive/EXPORT_FOR_COLAB/ML_RESULTS_UPGRADED"
DATA_DIR = "/content/drive/MyDrive/EXPORT_FOR_COLAB"

X_FP   = os.path.join(DATA_DIR, "X_samples_x_genes.tsv.gz")
Y_FP   = os.path.join(DATA_DIR, "y_labels.tsv")
PAM_FP = os.path.join(OUT_DIR, "pam50_genes.tsv")

def smart_read_table(fp):
    return pd.read_csv(fp, sep=None, engine="python")

def clean_genes(iterable):
    s = pd.Series(list(iterable)).astype(str).str.upper().str.strip()
    s = s.replace({"NAN":"", "NA":"", "N/A":"", "NONE":"", "NULL":"", "INF":"", "-INF":""})
    s = s[s != ""]
    s = s[s.str.match(r"^[A-Z0-9\-]+$")]
    s = s[s.str.len() > 1]
    return sorted(set(s.tolist()))

def pick_stable_top80(out_dir: str):
    explicit = [
        os.path.join(out_dir, "stable_compact_panel_TOP80_5seeds_5.csv"),
        os.path.join(out_dir, "stable_compact_panel_TOP80_5seeds.csv"),
        os.path.join(out_dir, "descriptive_TOP80_by_inclusion_frequency_5seeds_5.csv"),
        os.path.join(out_dir, "descriptive_TOP80_by_inclusion_frequency_5seeds.csv"),
        os.path.join(out_dir, "descriptive_gene_inclusion_frequency_TOP80_5seeds_5.csv"),
        os.path.join(out_dir, "descriptive_gene_inclusion_frequency_TOP80_5seeds.csv"),
    ]
    for fp in explicit:
        if os.path.exists(fp):
            return fp

    hits = sorted(glob.glob(os.path.join(out_dir, "*TOP80*.csv"))) + \
           sorted(glob.glob(os.path.join(out_dir, "*TOP80*.tsv")))
    if not hits:
        hits2 = sorted(glob.glob(os.path.join(out_dir, "*TOP80*")))
        raise FileNotFoundError("No TOP80-like file found. Found:\n" + "\n".join(hits2[:80]))

    def score(fp):
        n = os.path.basename(fp).lower()
        s = 0
        if "stable_compact_panel" in n: s += 100
        if "descriptive_top80_by_inclusion_frequency" in n: s += 60
        if "descriptive_gene_inclusion_frequency" in n: s += 30
        if "5seeds" in n: s += 10
        if n.endswith(".csv") or n.endswith(".tsv"): s += 5
        return s

    return sorted(hits, key=score, reverse=True)[0]

# ---- locate stable TOP80 ----
MY80_FP = pick_stable_top80(OUT_DIR)
print("✅ Using stable TOP80:", MY80_FP)

# ---- hard checks ----
for fp in [X_FP, Y_FP, PAM_FP, MY80_FP]:
    if not os.path.exists(fp):
        raise FileNotFoundError(f"Missing required file: {fp}")

# ---- load + clean gene lists ----
my80_raw = smart_read_table(MY80_FP)
my80_col = "GENE" if "GENE" in my80_raw.columns else my80_raw.columns[0]
my80 = clean_genes(my80_raw[my80_col].tolist())

pam_raw = pd.read_csv(PAM_FP, sep="\t")
pam_col = "GENE" if "GENE" in pam_raw.columns else pam_raw.columns[0]
pam = clean_genes(pam_raw[pam_col].tolist())

overlap = sorted(set(my80) & set(pam))
non_overlap = sorted(set(my80) - set(pam))

print("TOP80 (clean):", len(my80))
print("PAM50 (clean):", len(pam))
print("Overlap:", len(overlap), overlap)
print("Non-overlap:", len(non_overlap))

ov_fp  = os.path.join(OUT_DIR, "TOP80_overlap_with_PAM50.csv")
non_fp = os.path.join(OUT_DIR, "TOP80_nonoverlap_vs_PAM50.csv")
pd.Series(overlap, name="GENE").to_csv(ov_fp, index=False)
pd.Series(non_overlap, name="GENE").to_csv(non_fp, index=False)
print("Saved:", ov_fp)
print("Saved:", non_fp)

# ---- enrichment on non-overlap (Enrichr via gseapy) ----
try:
    import gseapy as gp
except Exception:
    !pip -q install gseapy
    import gseapy as gp

libs = ["Reactome_2022", "KEGG_2021_Human", "GO_Biological_Process_2023"]

all_res = []
for lib in libs:
    try:
        r = gp.enrichr(
            gene_list=non_overlap,
            gene_sets=lib,
            organism="human",   # FIXED
            outdir=None
        )
        df_res = r.results.copy()
        df_res["library"] = lib
        all_res.append(df_res)
        print(f"✅ Enrichment done for: {lib} | rows={df_res.shape[0]}")
    except Exception as e:
        print(f"⚠️ Failed enrichment for {lib}: {e}")

if len(all_res) == 0:
    raise RuntimeError("All enrichment libraries failed. Check internet connection / gseapy / Enrichr availability.")

enr = pd.concat(all_res, ignore_index=True)

sort_col = "Adjusted P-value" if "Adjusted P-value" in enr.columns else (
    "P-value" if "P-value" in enr.columns else None
)
if sort_col is not None:
    enr = enr.sort_values(sort_col).reset_index(drop=True)

enr_fp = os.path.join(OUT_DIR, "enrichment_TOP80_nonoverlap_enrichr.csv")
enr.to_csv(enr_fp, index=False)
print("Saved:", enr_fp)
display(enr.head(20))

# ---- multinomial LR coefficients on TOP80 (4-class subset, seed=123 split) ----
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

y_df = pd.read_csv(Y_FP, sep="\t")
if not {"SAMPLE_ID","y"}.issubset(y_df.columns):
    raise ValueError("y_labels.tsv must contain SAMPLE_ID and y")

X_df = pd.read_csv(X_FP, sep="\t", compression="gzip")
if "SAMPLE_ID" not in X_df.columns:
    raise ValueError("X_samples_x_genes.tsv.gz must contain SAMPLE_ID")

df = X_df.merge(y_df, on="SAMPLE_ID", how="inner")
feature_cols = [c for c in df.columns if c not in ("SAMPLE_ID","y")]
y_str = df["y"].astype(str).to_numpy()

CLASS_ORDER = ["Basal", "Her2", "LumA", "LumB"]
mask4 = np.isin(y_str, np.array(CLASS_ORDER, dtype=object))
df4 = df.loc[mask4].copy()
print("4-class counts:", df4["y"].value_counts().to_dict())

gene_to_col = {c.upper().strip(): c for c in feature_cols}
present = [g for g in my80 if g in gene_to_col]
missing_genes = [g for g in my80 if g not in gene_to_col]

print("TOP80 present in X:", len(present))
if missing_genes:
    print("Missing TOP80 genes (first 25):", missing_genes[:25])

X80 = df4[[gene_to_col[g] for g in present]].to_numpy(dtype=np.float32)
genes80 = present

lab_map = {c:i for i,c in enumerate(CLASS_ORDER)}
y4 = np.array([lab_map[s] for s in df4["y"].astype(str).to_numpy()], dtype=int)

SEED = 123
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
tr_idx, tmp_idx = next(sss1.split(X80, y4))
Xtr_raw, ytr = X80[tr_idx], y4[tr_idx]
Xtmp_raw, ytmp = X80[tmp_idx], y4[tmp_idx]

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
va_rel, te_rel = next(sss2.split(Xtmp_raw, ytmp))
va_idx, te_idx = tmp_idx[va_rel], tmp_idx[te_rel]
Xva_raw, yva = X80[va_idx], y4[va_idx]
Xte_raw, yte = X80[te_idx], y4[te_idx]

imp = SimpleImputer(strategy="median")
sc  = StandardScaler()
Xtr = sc.fit_transform(imp.fit_transform(Xtr_raw))
Xva = sc.transform(imp.transform(Xva_raw))
Xte = sc.transform(imp.transform(Xte_raw))

lr = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    class_weight="balanced",
    max_iter=6000,
    multi_class="multinomial",
    random_state=SEED
)
lr.fit(Xtr, ytr)

coef_df = pd.DataFrame(lr.coef_.T, index=genes80, columns=CLASS_ORDER).reset_index().rename(columns={"index":"GENE"})
coef_fp = os.path.join(OUT_DIR, "TOP80_multinomialLR_coefficients_4class.csv")
coef_df.to_csv(coef_fp, index=False)
print("Saved:", coef_fp)

TOPN = 15
rows = []
for c in CLASS_ORDER:
    tmp = coef_df[["GENE", c]].sort_values(c, ascending=False).head(TOPN)
    tmp = tmp.assign(class_name=c, rank=np.arange(1, len(tmp)+1))
    rows.append(tmp.rename(columns={c:"coef"}))

top_table = pd.concat(rows, ignore_index=True)[["class_name","rank","GENE","coef"]]
top_fp = os.path.join(OUT_DIR, f"TOP80_top{TOPN}_genes_per_class_4class.csv")
top_table.to_csv(top_fp, index=False)
print("Saved:", top_fp)
display(top_table)

In [ ]:
# ============================================================
# Cell 24 — FIXED: Refit multinomial LR on STABLE TOP80
# using PRIMARY 4-class split by SAMPLE_IDs (seed=123)
#
# Reads split from:
#   OUT_DIR/primary4_split_seed123.npz
# Keys expected:
#   tr_sid, va_sid, te_sid, class_order, seed
#
# Outputs (OUT_DIR):
#   - TOP80_refit_performance_seed123.csv
#   - TOP80_multinomialLR_coefficients_4class_refit.csv
#   - TOP80_top15_genes_per_class_4class_refit.csv
# ============================================================

import os, glob
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

# -----------------------
# Paths
# -----------------------
DATA_DIR = "/content/drive/MyDrive/EXPORT_FOR_COLAB"
OUT_DIR  = "/content/drive/MyDrive/EXPORT_FOR_COLAB/ML_RESULTS_UPGRADED"

X_FP     = os.path.join(DATA_DIR, "X_samples_x_genes.tsv.gz")
Y_FP     = os.path.join(DATA_DIR, "y_labels.tsv")
SPLIT_FP = os.path.join(OUT_DIR, "primary4_split_seed123.npz")  # ✅ FIXED

for fp in [X_FP, Y_FP, SPLIT_FP]:
    if not os.path.exists(fp):
        raise FileNotFoundError(f"Missing required file: {fp}")

# -----------------------
# Robust stable TOP80 discovery (handles *_5.csv variants)
# -----------------------
stable_candidates = [
    os.path.join(OUT_DIR, "stable_compact_panel_TOP80_5seeds_5.csv"),
    os.path.join(OUT_DIR, "stable_compact_panel_TOP80_5seeds.csv"),
    os.path.join(OUT_DIR, "descriptive_TOP80_by_inclusion_frequency_5seeds_5.csv"),
    os.path.join(OUT_DIR, "descriptive_TOP80_by_inclusion_frequency_5seeds.csv"),
    os.path.join(OUT_DIR, "descriptive_gene_inclusion_frequency_TOP80_5seeds_5.csv"),
    os.path.join(OUT_DIR, "descriptive_gene_inclusion_frequency_TOP80_5seeds.csv"),
]
MY80_FP = next((fp for fp in stable_candidates if os.path.exists(fp)), None)
if MY80_FP is None:
    hits = sorted(glob.glob(os.path.join(OUT_DIR, "*TOP80*")))
    raise FileNotFoundError("Stable TOP80 list not found. Found:\n" + "\n".join(hits[:80]))

print("✅ Using stable TOP80:", MY80_FP)

# -----------------------
# Helpers
# -----------------------
def clean_genes(iterable):
    s = pd.Series(list(iterable)).astype(str).str.upper().str.strip()
    s = s.replace({"NAN":"", "NA":"", "N/A":"", "NONE":"", "NULL":"", "INF":"", "-INF":""})
    s = s[s != ""]
    s = s[s.str.match(r"^[A-Z0-9\-]+$")]
    s = s[s.str.len() > 1]
    return sorted(set(s.tolist()))

# -----------------------
# Load aligned data (X + y)
# -----------------------
y_df = pd.read_csv(Y_FP, sep="\t")
if not {"SAMPLE_ID","y"}.issubset(y_df.columns):
    raise ValueError("y_labels.tsv must contain SAMPLE_ID and y columns")

X_df = pd.read_csv(X_FP, sep="\t", compression="gzip")
if "SAMPLE_ID" not in X_df.columns:
    raise ValueError("X_samples_x_genes.tsv.gz must contain SAMPLE_ID column")

df = X_df.merge(y_df, on="SAMPLE_ID", how="inner")
feature_cols = [c for c in df.columns if c not in ("SAMPLE_ID","y")]

# -----------------------
# Load split (PRIMARY 4-class) + enforce class order from NPZ
# -----------------------
spl = np.load(SPLIT_FP, allow_pickle=True)
need_keys = {"tr_sid","va_sid","te_sid","class_order","seed"}
if not need_keys.issubset(set(spl.files)):
    raise ValueError(f"primary4_split_seed123.npz missing keys. Found: {spl.files}")

CLASS_ORDER = spl["class_order"].astype(str).tolist()
print("✅ Class order from split:", CLASS_ORDER)

# subset to these 4 classes only
df4 = df[df["y"].astype(str).isin(CLASS_ORDER)].copy()
df4["SAMPLE_ID"] = df4["SAMPLE_ID"].astype(str)
df4["y"] = df4["y"].astype(str)

if df4.shape[0] == 0:
    raise RuntimeError("No samples found for the 4-class subset. Check label names in y_labels.tsv.")

lab_map = {c:i for i,c in enumerate(CLASS_ORDER)}
y4 = np.array([lab_map[s] for s in df4["y"].to_numpy()], dtype=int)

# -----------------------
# Load + clean TOP80 list
# -----------------------
tmp80 = pd.read_csv(MY80_FP, sep=None, engine="python")
gene_col = "GENE" if "GENE" in tmp80.columns else tmp80.columns[0]
my80 = clean_genes(tmp80[gene_col].tolist())
print("TOP80 (clean unique):", len(my80))

# Map to expression matrix columns (case-insensitive)
gene_to_col = {c.upper().strip(): c for c in feature_cols}
present = [g for g in my80 if g in gene_to_col]
missing_genes = [g for g in my80 if g not in gene_to_col]

print(f"TOP80 present in X: {len(present)} / {len(my80)}")
if missing_genes:
    print("⚠️ Missing TOP80 genes (first 30):", missing_genes[:30])

if len(present) < 10:
    raise RuntimeError(
        f"Too few TOP80 genes matched expression columns ({len(present)}). "
        "Likely gene naming mismatch (symbols vs Ensembl)."
    )

genes80 = present
X80_all = df4[[gene_to_col[g] for g in genes80]].to_numpy(dtype=np.float32)
sid4 = df4["SAMPLE_ID"].to_numpy(dtype=str)

# -----------------------
# Apply split by SAMPLE_IDs
# -----------------------
tr_sid = spl["tr_sid"].astype(str)
va_sid = spl["va_sid"].astype(str)
te_sid = spl["te_sid"].astype(str)

sid_to_idx = {s:i for i,s in enumerate(sid4)}

def map_sids(sids, name):
    miss = [s for s in sids if s not in sid_to_idx]
    if miss:
        raise RuntimeError(f"{name}: split SAMPLE_IDs not found in current df4 (first 10): {miss[:10]}")
    return np.array([sid_to_idx[s] for s in sids], dtype=int)

tr_idx = map_sids(tr_sid, "train")
va_idx = map_sids(va_sid, "val")
te_idx = map_sids(te_sid, "test")

print("✅ Split sizes:", len(tr_idx), len(va_idx), len(te_idx))

Xtr_raw, ytr = X80_all[tr_idx], y4[tr_idx]
Xva_raw, yva = X80_all[va_idx], y4[va_idx]
Xte_raw, yte = X80_all[te_idx], y4[te_idx]

# -----------------------
# Train-only impute + scale
# -----------------------
imp = SimpleImputer(strategy="median")
sc  = StandardScaler()
Xtr = sc.fit_transform(imp.fit_transform(Xtr_raw))
Xva = sc.transform(imp.transform(Xva_raw))
Xte = sc.transform(imp.transform(Xte_raw))

# -----------------------
# Fit multinomial LR
# -----------------------
lr = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    class_weight="balanced",
    max_iter=6000,
    multi_class="multinomial",
    random_state=int(spl["seed"])
)
lr.fit(Xtr, ytr)

# -----------------------
# Evaluate VAL + TEST
# -----------------------
def eval_split(name, X, y_true):
    y_pred = lr.predict(X)
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro")
    f1w = f1_score(y_true, y_pred, average="weighted")
    cm  = confusion_matrix(y_true, y_pred, labels=np.arange(len(CLASS_ORDER)))
    print(f"{name.upper():>4} | acc={acc:.4f} | macro-F1={f1m:.4f} | weighted-F1={f1w:.4f}")
    return {"split": name, "n": int(len(y_true)), "accuracy": float(acc), "macro_f1": float(f1m), "weighted_f1": float(f1w), "cm": cm}

val_res  = eval_split("val",  Xva, yva)
test_res = eval_split("test", Xte, yte)

perf_df = pd.DataFrame([
    {k:v for k,v in val_res.items()  if k != "cm"},
    {k:v for k,v in test_res.items() if k != "cm"},
])
perf_fp = os.path.join(OUT_DIR, "TOP80_refit_performance_seed123.csv")
perf_df.to_csv(perf_fp, index=False)
print("✅ Saved:", perf_fp)

# -----------------------
# Export coefficients (genes x classes)
# -----------------------
coef_df = pd.DataFrame(lr.coef_.T, index=genes80, columns=CLASS_ORDER).reset_index().rename(columns={"index":"GENE"})
coef_fp = os.path.join(OUT_DIR, "TOP80_multinomialLR_coefficients_4class_refit.csv")
coef_df.to_csv(coef_fp, index=False)
print("✅ Saved:", coef_fp)

# -----------------------
# Top genes per class
# -----------------------
TOPN = 15
rows = []
for c in CLASS_ORDER:
    tmp = coef_df[["GENE", c]].sort_values(c, ascending=False).head(TOPN)
    tmp = tmp.assign(class_name=c, rank=np.arange(1, len(tmp)+1))
    rows.append(tmp.rename(columns={c: "coef"}))

top_table = pd.concat(rows, ignore_index=True)[["class_name","rank","GENE","coef"]]
top_fp = os.path.join(OUT_DIR, f"TOP80_top{TOPN}_genes_per_class_4class_refit.csv")
top_table.to_csv(top_fp, index=False)
print("✅ Saved:", top_fp)

display(top_table.head(60))

In [ ]:
# ============================================================
# Cell 25 — FIXED: SAFE refit multinomial LR on STABLE TOP80
# using PRIMARY 4-class split by SAMPLE_IDs (seed=123)
# Split file:
#   primary4_split_seed123.npz
# Exports:
#   - STABLE_TOP80_multinomialLR_coefficients_4class_seed123split.csv
#   - STABLE_TOP80_top15_genes_per_class_4class_seed123split.csv
# ============================================================

import os, glob
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

DATA_DIR = "/content/drive/MyDrive/EXPORT_FOR_COLAB"
OUT_DIR  = "/content/drive/MyDrive/EXPORT_FOR_COLAB/ML_RESULTS_UPGRADED"

X_FP     = os.path.join(DATA_DIR, "X_samples_x_genes.tsv.gz")
Y_FP     = os.path.join(DATA_DIR, "y_labels.tsv")
SPLIT_FP = os.path.join(OUT_DIR, "primary4_split_seed123.npz")  # ✅ FIXED

# stable TOP80 discovery
stable_candidates = [
    os.path.join(OUT_DIR, "stable_compact_panel_TOP80_5seeds_5.csv"),
    os.path.join(OUT_DIR, "stable_compact_panel_TOP80_5seeds.csv"),
    os.path.join(OUT_DIR, "descriptive_TOP80_by_inclusion_frequency_5seeds_5.csv"),
    os.path.join(OUT_DIR, "descriptive_TOP80_by_inclusion_frequency_5seeds.csv"),
]
MY80_FP = next((fp for fp in stable_candidates if os.path.exists(fp)), None)
if MY80_FP is None:
    hits = sorted(glob.glob(os.path.join(OUT_DIR, "*TOP80*")))
    raise FileNotFoundError("Stable TOP80 list not found. Found:\n" + "\n".join(hits[:60]))

for fp in [X_FP, Y_FP, MY80_FP, SPLIT_FP]:
    if not os.path.exists(fp):
        raise FileNotFoundError(f"Missing required file: {fp}")

print("✅ Using stable TOP80:", MY80_FP)
print("✅ Using split:", SPLIT_FP)

def clean_genes(iterable):
    s = pd.Series(list(iterable)).astype(str).str.upper().str.strip()
    s = s.replace({"NAN":"", "NA":"", "N/A":"", "NONE":"", "NULL":"", "INF":"", "-INF":""})
    s = s[s != ""]
    s = s[s.str.match(r"^[A-Z0-9\-]+$")]
    s = s[s.str.len() > 1]
    return sorted(set(s.tolist()))

# Load aligned X+y
y_df = pd.read_csv(Y_FP, sep="\t")
X_df = pd.read_csv(X_FP, sep="\t", compression="gzip")
df = X_df.merge(y_df, on="SAMPLE_ID", how="inner")
feature_cols = [c for c in df.columns if c not in ("SAMPLE_ID","y")]

# Load split (primary4)
spl = np.load(SPLIT_FP, allow_pickle=True)
CLASS_ORDER = spl["class_order"].astype(str).tolist()

df4 = df[df["y"].astype(str).isin(CLASS_ORDER)].copy()
df4["SAMPLE_ID"] = df4["SAMPLE_ID"].astype(str)
df4["y"] = df4["y"].astype(str)

lab_map = {c:i for i,c in enumerate(CLASS_ORDER)}
y4 = np.array([lab_map[s] for s in df4["y"].to_numpy()], dtype=int)

# Load stable TOP80 genes
my80_raw = pd.read_csv(MY80_FP, sep=None, engine="python")
my80_col = "GENE" if "GENE" in my80_raw.columns else my80_raw.columns[0]
my80 = clean_genes(my80_raw[my80_col].tolist())

gene_to_col = {c.upper().strip(): c for c in feature_cols}
present = [g for g in my80 if g in gene_to_col]
missing = [g for g in my80 if g not in gene_to_col]

print("Stable TOP80 present in X:", len(present), "/", len(my80))
if missing:
    print("⚠️ Missing genes (first 25):", missing[:25])
if len(present) < 10:
    raise RuntimeError("Too few TOP80 genes matched expression columns. Check gene naming.")

X80 = df4[[gene_to_col[g] for g in present]].to_numpy(dtype=np.float32)
genes80 = present
sid4 = df4["SAMPLE_ID"].to_numpy(dtype=str)

# Apply split by SAMPLE_IDs
tr_sid = spl["tr_sid"].astype(str)
va_sid = spl["va_sid"].astype(str)
te_sid = spl["te_sid"].astype(str)

sid_to_idx = {s:i for i,s in enumerate(sid4)}
def map_sids(sids, name):
    miss = [s for s in sids if s not in sid_to_idx]
    if miss:
        raise RuntimeError(f"{name}: split SAMPLE_IDs not found (first 10): {miss[:10]}")
    return np.array([sid_to_idx[s] for s in sids], dtype=int)

tr_idx = map_sids(tr_sid, "train")
va_idx = map_sids(va_sid, "val")
te_idx = map_sids(te_sid, "test")
print("Split sizes:", len(tr_idx), len(va_idx), len(te_idx))

Xtr_raw, ytr = X80[tr_idx], y4[tr_idx]
Xva_raw, yva = X80[va_idx], y4[va_idx]
Xte_raw, yte = X80[te_idx], y4[te_idx]

# Train-only impute + scale
imp = SimpleImputer(strategy="median")
sc  = StandardScaler()
Xtr = sc.fit_transform(imp.fit_transform(Xtr_raw))
Xva = sc.transform(imp.transform(Xva_raw))
Xte = sc.transform(imp.transform(Xte_raw))

# Multinomial LR
lr = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    class_weight="balanced",
    max_iter=6000,
    multi_class="multinomial",
    random_state=int(spl["seed"])
)
lr.fit(Xtr, ytr)

coef_df = pd.DataFrame(lr.coef_.T, index=genes80, columns=CLASS_ORDER).reset_index().rename(columns={"index":"GENE"})
coef_fp = os.path.join(OUT_DIR, "STABLE_TOP80_multinomialLR_coefficients_4class_seed123split.csv")
coef_df.to_csv(coef_fp, index=False)
print("✅ Saved:", coef_fp)

TOPN = 15
rows = []
for c in CLASS_ORDER:
    tmp = coef_df[["GENE", c]].sort_values(c, ascending=False).head(TOPN)
    tmp = tmp.assign(class_name=c, rank=np.arange(1, len(tmp)+1))
    rows.append(tmp.rename(columns={c: "coef"}))

top_table = pd.concat(rows, ignore_index=True)[["class_name","rank","GENE","coef"]]
top_fp = os.path.join(OUT_DIR, f"STABLE_TOP80_top{TOPN}_genes_per_class_4class_seed123split.csv")
top_table.to_csv(top_fp, index=False)
print("✅ Saved:", top_fp)

display(top_table.head(60))

In [ ]:
# ============================================================
# Cell 26 — Paper-ready visualization pack (BMC Bioinformatics style)
# Enhanced journal-style version
# Generates high-quality, informative figures from your saved CSV outputs.
# Saves PNG+PDF (700 dpi) into: OUT_DIR/FIGURES_4CLASS_PAPER/
#
# Figures (if inputs exist):
#  A) Robustness across 5 seeds (paired mean±SD) — from Cell 18 per-seed CSV
#  B) Confusion matrices (normalized) on TEST seed=123 — PAM50 vs Compact (from recompute)
#     + combined side-by-side journal figure
#  C) Macro ROC + PR curves (seed=123) — PAM50 vs Compact (recompute, publication-safe)
#  D) Stable TOP80 vs PAM50 overlap summary bar + gene list sizes
#  E) Enrichment dot-plot (non-overlap genes) — from Enrichr CSV (Cell 23)
#  F) Coefficient heatmap (stable TOP80 multinomial LR) — from coefficient CSV (Cell 24/25)
# ============================================================

import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, roc_curve, precision_recall_curve, auc,
    balanced_accuracy_score, f1_score
)

# -----------------------
# Paths
# -----------------------
DATA_DIR = "/content/drive/MyDrive/EXPORT_FOR_COLAB"
OUT_DIR  = os.path.join(DATA_DIR, "ML_RESULTS_UPGRADED")
FIG_DIR  = os.path.join(OUT_DIR, "FIGURES_4CLASS_PAPER")
os.makedirs(FIG_DIR, exist_ok=True)

X_FP      = os.path.join(DATA_DIR, "X_samples_x_genes.tsv.gz")
Y_FP      = os.path.join(DATA_DIR, "y_labels.tsv")
CENT_FP   = os.path.join(OUT_DIR, "pam50_centroids.tsv")

PER_SEED_FP = os.path.join(OUT_DIR, "robustness_pam50_vs_compactLR_4class_per_seed.csv")
ENR_FP      = os.path.join(OUT_DIR, "enrichment_TOP80_nonoverlap_enrichr.csv")
OV_FP       = os.path.join(OUT_DIR, "TOP80_overlap_with_PAM50.csv")
NON_FP      = os.path.join(OUT_DIR, "TOP80_nonoverlap_vs_PAM50.csv")

COEF_CANDIDATES = [
    os.path.join(OUT_DIR, "STABLE_TOP80_multinomialLR_coefficients_4class_seed123split.csv"),
    os.path.join(OUT_DIR, "TOP80_multinomialLR_coefficients_4class_refit.csv"),
    os.path.join(OUT_DIR, "TOP80_multinomialLR_coefficients_4class.csv"),
]
COEF_FP = next((fp for fp in COEF_CANDIDATES if os.path.exists(fp)), None)

# -----------------------
# Style (journal-friendly)
# -----------------------
plt.rcParams.update({
    "figure.dpi": 170,
    "savefig.dpi": 700,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 1.0,
})

GRID_COLOR = "#D6D6D6"
TEXT_COLOR = "#222222"
PAM50_COLOR = "#1A1A1A"
COMPACT_COLOR = "#0B7285"
MID_GREY = "#9A9A9A"
LIGHT_GREY = "#B8B8B8"

def savefig(fig, name):
    png = os.path.join(FIG_DIR, f"{name}.png")
    pdf = os.path.join(FIG_DIR, f"{name}.pdf")
    fig.savefig(png, bbox_inches="tight", facecolor="white", dpi=700)
    fig.savefig(pdf, bbox_inches="tight", facecolor="white", dpi=700)
    print("Saved:", png)
    print("Saved:", pdf)

def mean_sd(x):
    x = np.asarray(x, dtype=float)
    return float(x.mean()), float(x.std(ddof=0))

def softmax(S):
    S = S - S.max(axis=1, keepdims=True)
    ex = np.exp(S)
    return ex / (ex.sum(axis=1, keepdims=True) + 1e-12)

# ============================================================
# FIG A — Robustness (TEST): paired + mean±SD across 5 seeds
# Enhanced journal-style version
# ============================================================
if os.path.exists(PER_SEED_FP):
    per_seed = pd.read_csv(PER_SEED_FP)

    METRICS = [
        ("macro_f1", "Macro-F1"),
        ("macro_roc_auc_ovr", "ROC-AUC\n(macro OVR)"),
        ("macro_pr_auc", "PR-AUC\n(macro)"),
        ("bacc", "Balanced\naccuracy"),
    ]

    rows = []
    for key, lab in METRICS:
        p = per_seed[f"pam50_test_{key}"].to_numpy(dtype=float)
        c = per_seed[f"compact_test_{key}"].to_numpy(dtype=float)
        pm, ps = mean_sd(p)
        cm, cs = mean_sd(c)
        dm, ds = mean_sd(c - p)
        rows.append([key, lab, pm, ps, cm, cs, dm, ds])

    plot_df = pd.DataFrame(
        rows,
        columns=[
            "key", "metric",
            "pam50_mean", "pam50_sd",
            "compact_mean", "compact_sd",
            "delta_mean", "delta_sd"
        ]
    )

    x = np.arange(len(plot_df))
    dx = 0.14

    fig, ax = plt.subplots(figsize=(10.8, 4.8))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    ax.grid(True, axis="y", lw=0.7, color=GRID_COLOR, alpha=0.65, zorder=0)
    ax.grid(False, axis="x")

    for i, (key, _) in enumerate(METRICS):
        p = per_seed[f"pam50_test_{key}"].to_numpy(dtype=float)
        c = per_seed[f"compact_test_{key}"].to_numpy(dtype=float)

        for j in range(len(p)):
            ax.plot(
                [i - dx, i + dx], [p[j], c[j]],
                lw=1.0, color=LIGHT_GREY, alpha=0.95, zorder=1
            )

        ax.scatter(
            np.full_like(p, i - dx, dtype=float), p,
            s=28, color=PAM50_COLOR, alpha=0.18, zorder=2
        )
        ax.scatter(
            np.full_like(c, i + dx, dtype=float), c,
            s=28, color=COMPACT_COLOR, alpha=0.18, zorder=2
        )

    ax.errorbar(
        x - dx,
        plot_df["pam50_mean"],
        yerr=plot_df["pam50_sd"],
        fmt="o",
        markersize=7.0,
        lw=1.5,
        capsize=4,
        capthick=1.2,
        color=PAM50_COLOR,
        ecolor=PAM50_COLOR,
        label="PAM50 centroid",
        zorder=4
    )

    ax.errorbar(
        x + dx,
        plot_df["compact_mean"],
        yerr=plot_df["compact_sd"],
        fmt="o",
        markersize=7.0,
        lw=1.5,
        capsize=4,
        capthick=1.2,
        color=COMPACT_COLOR,
        ecolor=COMPACT_COLOR,
        label="Compact LR (TOP80)",
        zorder=4
    )

    for i in range(len(plot_df)):
        pm = float(plot_df.loc[i, "pam50_mean"])
        ps = float(plot_df.loc[i, "pam50_sd"])
        cm = float(plot_df.loc[i, "compact_mean"])
        cs = float(plot_df.loc[i, "compact_sd"])
        dm = float(plot_df.loc[i, "delta_mean"])

        top = max(pm + ps, cm + cs)

        if top > 0.94:
            y_annot = top + 0.010
        elif top > 0.86:
            y_annot = top + 0.018
        else:
            y_annot = top + 0.028

        ax.text(
            i,
            y_annot,
            f"Δ = {dm:+.3f}",
            ha="center",
            va="bottom",
            fontsize=10.5,
            color=TEXT_COLOR,
            bbox=dict(
                boxstyle="round,pad=0.22",
                facecolor="white",
                edgecolor="#CFCFCF",
                lw=0.8
            ),
            zorder=5
        )

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["metric"], rotation=0, ha="center", color=TEXT_COLOR)
    ax.set_ylabel("Test score", color=TEXT_COLOR)
    ax.set_ylim(0.0, 1.03)
    ax.set_xlim(-0.3, len(plot_df) - 0.7)

    ax.set_title(
        "Robustness across 5 random seeds: Compact LR versus PAM50",
        color=TEXT_COLOR,
        pad=14
    )

    ax.tick_params(axis="x", colors=TEXT_COLOR, pad=6)
    ax.tick_params(axis="y", colors=TEXT_COLOR)
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_linewidth(1.0)
        ax.spines[spine].set_color(TEXT_COLOR)

    ax.legend(
        loc="lower right",
        frameon=False,
        handletextpad=0.8,
        borderpad=0.2
    )

    ax.text(
        -0.06, 1.04, "A",
        transform=ax.transAxes,
        fontsize=13, fontweight="bold",
        ha="left", va="bottom", color=TEXT_COLOR
    )

    savefig(fig, "FIGA_robustness_paired_point_range_meanSD_TEST")
    plt.show()
else:
    print("FIG A skipped: missing", PER_SEED_FP)

# ============================================================
# Recompute seed=123 TEST probs (publication-safe) for FIG B + C
# ============================================================
RECOMPUTE_OK = all(os.path.exists(fp) for fp in [X_FP, Y_FP, CENT_FP])
if not RECOMPUTE_OK:
    print("FIG B/C skipped: missing one of X_FP/Y_FP/CENT_FP")
else:
    SEED = 123
    CLASS_ORDER = ["Basal", "Her2", "LumA", "LumB"]
    lab_map = {c:i for i,c in enumerate(CLASS_ORDER)}
    K = 4

    y_df = pd.read_csv(Y_FP, sep="\t")
    X_df = pd.read_csv(X_FP, sep="\t", compression="gzip")
    df = X_df.merge(y_df, on="SAMPLE_ID", how="inner")
    feature_cols = [c for c in df.columns if c not in ("SAMPLE_ID", "y")]
    X_mat = df[feature_cols].to_numpy(dtype=np.float32)
    y_str = df["y"].astype(str).to_numpy()

    mask4 = np.isin(y_str, np.array(CLASS_ORDER, dtype=object))
    X4 = X_mat[mask4]
    y4_str = y_str[mask4]
    y4 = np.array([lab_map[s] for s in y4_str], dtype=int)

    all_idx = np.arange(len(y4))
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
    tr_idx, tmp_idx = next(sss1.split(all_idx, y4))
    ytmp = y4[tmp_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
    va_rel, te_rel = next(sss2.split(tmp_idx, ytmp))
    va_idx = tmp_idx[va_rel]
    te_idx = tmp_idx[te_rel]

    Xtr_raw, ytr = X4[tr_idx], y4[tr_idx]
    Xva_raw, yva = X4[va_idx], y4[va_idx]
    Xte_raw, yte = X4[te_idx], y4[te_idx]

    # ---- PAM50 centroids ----
    cent = pd.read_csv(CENT_FP, sep="\t")
    cent["GENE"] = cent["GENE"].astype(str).str.upper().str.strip()
    centroids_df = cent.set_index("GENE")[CLASS_ORDER]

    ALIAS_MAP = {"CDCA1": "NUF2", "KNTC2": "NDC80", "ORC6L": "ORC6"}
    genes_full = np.array(feature_cols, dtype=str)
    genes_full_U = np.char.upper(np.char.strip(genes_full))
    gene_to_idx = {g: i for i, g in enumerate(genes_full_U)}

    pam_genes = centroids_df.index.to_list()
    used_pairs, idx_pam = [], []
    for g in pam_genes:
        gU = str(g).upper().strip()
        if gU in gene_to_idx:
            idx_pam.append(gene_to_idx[gU])
            used_pairs.append((gU, gU))
        else:
            a = ALIAS_MAP.get(gU, None)
            if a is not None:
                aU = a.upper().strip()
                if aU in gene_to_idx:
                    idx_pam.append(gene_to_idx[aU])
                    used_pairs.append((gU, aU))

    idx_pam = np.array(idx_pam, dtype=int)
    if len(idx_pam) != 50:
        raise RuntimeError(f"PAM50 overlap not 50/50 (got {len(idx_pam)}). Check gene naming/aliases.")

    C_pam = centroids_df.loc[[p[0] for p in used_pairs], :].to_numpy(dtype=np.float32)

    def robust_center_scale_train(X_train, X):
        med = np.median(X_train, axis=0)
        mad = np.median(np.abs(X_train - med), axis=0) + 1e-6
        return (X - med) / mad

    def corr_scores(X, C):
        Xc = X - X.mean(axis=1, keepdims=True)
        Cc = C - C.mean(axis=0, keepdims=True)
        Xn = Xc / (np.linalg.norm(Xc, axis=1, keepdims=True) + 1e-8)
        Cn = Cc / (np.linalg.norm(Cc, axis=0, keepdims=True) + 1e-8)
        return Xn @ Cn

    Xtr_pam = Xtr_raw[:, idx_pam]
    Xte_pam = Xte_raw[:, idx_pam]
    Xtr_cs = robust_center_scale_train(Xtr_pam, Xtr_pam)
    Xte_cs = robust_center_scale_train(Xtr_pam, Xte_pam)
    scores_te_pam = corr_scores(Xte_cs, C_pam)
    proba_te_pam = softmax(scores_te_pam)
    pred_te_pam = proba_te_pam.argmax(axis=1)

    # ---- Compact LR probs on TEST ----
    V = int(min(3000, Xtr_raw.shape[1]))
    imp_rank = SimpleImputer(strategy="median")
    Xtr_imp = imp_rank.fit_transform(Xtr_raw)
    var = np.var(Xtr_imp, axis=0)
    v_idx = np.argsort(var)[::-1][:V]

    Xtr_v = Xtr_raw[:, v_idx]
    Xva_v = Xva_raw[:, v_idx]
    Xte_v = Xte_raw[:, v_idx]

    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()
    Xtr = sc.fit_transform(imp.fit_transform(Xtr_v)).astype(np.float32)
    Xva = sc.transform(imp.transform(Xva_v)).astype(np.float32)
    Xte = sc.transform(imp.transform(Xte_v)).astype(np.float32)

    PANEL_SIZE = 80
    C_SEL = 0.6

    sel = LogisticRegression(
        penalty="l1", solver="saga", C=float(C_SEL),
        class_weight="balanced", max_iter=1500, tol=1e-3,
        n_jobs=1, random_state=SEED
    )
    sel.fit(Xtr, ytr)
    score = np.mean(np.abs(sel.coef_), axis=0)
    panel_idx = np.argsort(score)[::-1][:PANEL_SIZE]

    Xtr_p = Xtr[:, panel_idx]
    Xva_p = Xva[:, panel_idx]
    Xte_p = Xte[:, panel_idx]

    C_TUNE = [0.3, 1.0, 3.0]
    bestC, bestF = None, -1.0
    for Cc in C_TUNE:
        m = LogisticRegression(
            penalty="l2", solver="lbfgs", C=float(Cc),
            class_weight="balanced", max_iter=4000,
            n_jobs=1, random_state=SEED
        )
        m.fit(Xtr_p, ytr)
        f = float(f1_score(yva, m.predict(Xva_p), average="macro"))
        if f > bestF:
            bestF, bestC = f, float(Cc)

    lr = LogisticRegression(
        penalty="l2", solver="lbfgs", C=float(bestC),
        class_weight="balanced", max_iter=4000,
        n_jobs=1, random_state=SEED
    )
    Xtrva_p = np.vstack([Xtr_p, Xva_p])
    ytrva = np.concatenate([ytr, yva])
    lr.fit(Xtrva_p, ytrva)

    proba_te_comp = lr.predict_proba(Xte_p)
    pred_te_comp = proba_te_comp.argmax(axis=1)

    # ============================================================
    # FIG B — Confusion matrices (TEST seed=123): PAM50 vs Compact
    # Includes individual figures + combined side-by-side figure
    # ============================================================
    def draw_cm(ax, cm, title):
        cm = cm.astype(float)
        cmn = cm / (cm.sum(axis=1, keepdims=True) + 1e-12)

        ax.set_facecolor("white")
        im = ax.imshow(cmn, vmin=0, vmax=1, cmap="Blues")

        ax.set_title(title, color=TEXT_COLOR, pad=10)
        ax.set_xlabel("Predicted", color=TEXT_COLOR)
        ax.set_ylabel("True", color=TEXT_COLOR)

        ax.set_xticks(np.arange(K))
        ax.set_yticks(np.arange(K))
        ax.set_xticklabels(CLASS_ORDER, rotation=35, ha="right", color=TEXT_COLOR)
        ax.set_yticklabels(CLASS_ORDER, color=TEXT_COLOR)

        for i in range(K):
            for j in range(K):
                ax.text(
                    j, i, f"{cmn[i, j]:.2f}",
                    ha="center", va="center", fontsize=10,
                    color=("white" if cmn[i, j] > 0.55 else TEXT_COLOR)
                )

        ax.tick_params(length=0)
        ax.grid(False)
        return im

    def plot_cm_single(cm, title, fname):
        fig, ax = plt.subplots(figsize=(5.4, 4.6))
        fig.patch.set_facecolor("white")

        im = draw_cm(ax, cm, title)

        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=9, colors=TEXT_COLOR)

        savefig(fig, fname)
        plt.show()

    def plot_cm_combined(cm_left, cm_right, title_left, title_right, fname):
        fig, axes = plt.subplots(
            1, 2, figsize=(10.8, 4.8),
            gridspec_kw={"wspace": 0.28}
        )
        fig.patch.set_facecolor("white")

        im = draw_cm(axes[0], cm_left, title_left)
        draw_cm(axes[1], cm_right, title_right)

        axes[0].text(
            -0.18, 1.06, "A",
            transform=axes[0].transAxes,
            fontsize=13, fontweight="bold",
            ha="left", va="bottom", color=TEXT_COLOR
        )
        axes[1].text(
            -0.18, 1.06, "B",
            transform=axes[1].transAxes,
            fontsize=13, fontweight="bold",
            ha="left", va="bottom", color=TEXT_COLOR
        )

        fig.subplots_adjust(right=0.88, wspace=0.32)
        cax = fig.add_axes([0.90, 0.16, 0.02, 0.68])
        cbar = fig.colorbar(im, cax=cax)
        cbar.ax.tick_params(labelsize=9, colors=TEXT_COLOR)

        savefig(fig, fname)
        plt.show()

    cm_p = confusion_matrix(yte, pred_te_pam, labels=np.arange(K))
    cm_c = confusion_matrix(yte, pred_te_comp, labels=np.arange(K))

    plot_cm_single(
        cm_p,
        "PAM50 centroid — TEST (seed=123)\nRow-normalized confusion",
        "FIGB_CM_PAM50_TEST_seed123"
    )

    plot_cm_single(
        cm_c,
        "Compact LR (TOP80) — TEST (seed=123)\nRow-normalized confusion",
        "FIGB_CM_CompactLR_TEST_seed123"
    )

    plot_cm_combined(
        cm_p,
        cm_c,
        "PAM50 centroid\nTEST (seed=123)",
        "Compact LR (TOP80)\nTEST (seed=123)",
        "FIGB_CM_PAM50_vs_Compact_TEST_seed123_combined"
    )

    # ============================================================
    # FIG C — Macro ROC + Macro PR curves (TEST seed=123)
    # ============================================================
    y_bin = label_binarize(yte, classes=np.arange(K))

    def macro_roc_curve(y_bin, prob, grid_n=600):
        grid = np.linspace(0, 1, grid_n)
        tprs = []
        for k in range(K):
            fpr, tpr, _ = roc_curve(y_bin[:, k], prob[:, k])
            tpr_i = np.interp(grid, fpr, tpr)
            tpr_i[0] = 0.0
            tprs.append(tpr_i)
        mean_tpr = np.mean(np.vstack(tprs), axis=0)
        mean_tpr[-1] = 1.0
        return grid, mean_tpr, auc(grid, mean_tpr)

    def macro_pr_curve(y_bin, prob, grid_n=600):
        grid = np.linspace(0, 1, grid_n)
        precs = []
        for k in range(K):
            prec, rec, _ = precision_recall_curve(y_bin[:, k], prob[:, k])
            order = np.argsort(rec)
            rec_s = rec[order]
            prec_s = prec[order]
            prec_i = np.interp(grid, rec_s, prec_s)
            precs.append(prec_i)
        mean_prec = np.mean(np.vstack(precs), axis=0)
        return grid, mean_prec, auc(grid, mean_prec)

    fpr_p, tpr_p, auc_p = macro_roc_curve(y_bin, proba_te_pam)
    fpr_c, tpr_c, auc_c = macro_roc_curve(y_bin, proba_te_comp)

    rec_p, prec_p, prauc_p = macro_pr_curve(y_bin, proba_te_pam)
    rec_c, prec_c, prauc_c = macro_pr_curve(y_bin, proba_te_comp)

    fig = plt.figure(figsize=(10.4, 4.4))
    fig.patch.set_facecolor("white")

    ax1 = fig.add_subplot(1, 2, 1)
    ax1.set_facecolor("white")
    ax1.plot(fpr_p, tpr_p, lw=2.0, color=PAM50_COLOR, label=f"PAM50 (macro AUC={auc_p:.3f})")
    ax1.plot(fpr_c, tpr_c, lw=2.0, color=COMPACT_COLOR, label=f"Compact LR (macro AUC={auc_c:.3f})")
    ax1.plot([0, 1], [0, 1], lw=1.0, color=MID_GREY, ls=":")
    ax1.set_xlabel("False positive rate", color=TEXT_COLOR)
    ax1.set_ylabel("True positive rate", color=TEXT_COLOR)
    ax1.set_title("ROC (macro OVR) — TEST (seed=123)", color=TEXT_COLOR, pad=10)
    ax1.grid(True, lw=0.6, color=GRID_COLOR, alpha=0.55)
    ax1.set_xlim(0, 1)
    ax1.set_ylim(0, 1)

    ax2 = fig.add_subplot(1, 2, 2)
    ax2.set_facecolor("white")
    ax2.plot(rec_p, prec_p, lw=2.0, color=PAM50_COLOR, label=f"PAM50 (macro AUC={prauc_p:.3f})")
    ax2.plot(rec_c, prec_c, lw=2.0, color=COMPACT_COLOR, label=f"Compact LR (macro AUC={prauc_c:.3f})")
    ax2.set_xlabel("Recall", color=TEXT_COLOR)
    ax2.set_ylabel("Precision", color=TEXT_COLOR)
    ax2.set_title("PR (macro) — TEST (seed=123)", color=TEXT_COLOR, pad=10)
    ax2.grid(True, lw=0.6, color=GRID_COLOR, alpha=0.55)
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)

    handles, labels = ax2.get_legend_handles_labels()
    fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)

    savefig(fig, "FIGC_macro_ROC_PR_seed123")
    plt.show()

# ============================================================
# FIG D — Stable TOP80 vs PAM50 overlap summary (bar)
# ============================================================
if os.path.exists(OV_FP) and os.path.exists(NON_FP):
    ov = pd.read_csv(OV_FP, sep=None, engine="python")
    non = pd.read_csv(NON_FP, sep=None, engine="python")
    n_ov = int(ov.shape[0])
    n_non = int(non.shape[0])
    n_tot = n_ov + n_non

    fig, ax = plt.subplots(figsize=(6.8, 4.0))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    ax.bar(
        ["Overlap\n(TOP80 ∩ PAM50)", "Non-overlap\n(TOP80 \\ PAM50)"],
        [n_ov, n_non],
        color=[PAM50_COLOR, COMPACT_COLOR],
        alpha=0.90,
        width=0.62
    )

    ax.set_ylabel("Gene count", color=TEXT_COLOR)
    ax.set_title("Stable TOP80 overlap with PAM50", color=TEXT_COLOR, pad=10)
    ax.grid(True, axis="y", lw=0.6, color=GRID_COLOR, alpha=0.55)

    for i, v in enumerate([n_ov, n_non]):
        ax.text(
            i, v + max(1, 0.02 * n_tot), f"{v}",
            ha="center", va="bottom", fontsize=11, color=TEXT_COLOR
        )

    savefig(fig, "FIGD_TOP80_overlap_summary")
    plt.show()
else:
    print("FIG D skipped: missing overlap/non-overlap CSVs")

# ============================================================
# FIG E — Enrichment dot-plot (non-overlap genes) from Enrichr
# ============================================================
if os.path.exists(ENR_FP):
    enr = pd.read_csv(ENR_FP, sep=None, engine="python")

    term_col = "Term" if "Term" in enr.columns else ("term" if "term" in enr.columns else None)
    lib_col = "library" if "library" in enr.columns else ("Gene_set" if "Gene_set" in enr.columns else None)
    p_col = "Adjusted P-value" if "Adjusted P-value" in enr.columns else ("P-value" if "P-value" in enr.columns else None)
    overlap_col = "Overlap" if "Overlap" in enr.columns else None

    if term_col is None or lib_col is None or p_col is None:
        print("FIG E skipped: enrichment file missing required columns. Found:", enr.columns.tolist())
    else:
        enr = enr.dropna(subset=[term_col, lib_col, p_col]).copy()
        enr[p_col] = pd.to_numeric(enr[p_col], errors="coerce")
        enr = enr.dropna(subset=[p_col]).copy()

        top = []
        for lib, sub in enr.groupby(lib_col):
            sub = sub.sort_values(p_col, ascending=True).head(8)
            top.append(sub)
        top = pd.concat(top, ignore_index=True)

        top["mlog10p"] = -np.log10(top[p_col].astype(float) + 1e-300)

        if overlap_col is not None:
            def parse_k(x):
                try:
                    return int(str(x).split("/")[0])
                except Exception:
                    return np.nan
            top["k"] = top[overlap_col].map(parse_k)
        else:
            top["k"] = np.nan

        top["_label"] = top[lib_col].astype(str) + " | " + top[term_col].astype(str)
        top = top.sort_values(["mlog10p"], ascending=True).tail(24).reset_index(drop=True)

        fig, ax = plt.subplots(figsize=(9.8, 5.9))
        fig.patch.set_facecolor("white")
        ax.set_facecolor("white")

        y = np.arange(len(top))
        size = np.where(np.isnan(top["k"]), 60, 30 + 15 * top["k"].astype(float))

        ax.scatter(
            top["mlog10p"], y,
            s=size, alpha=0.85,
            color=COMPACT_COLOR,
            edgecolors="black", linewidths=0.35
        )
        ax.set_yticks(y)
        ax.set_yticklabels(top["_label"], fontsize=9, color=TEXT_COLOR)
        ax.set_xlabel(r"$-\log_{10}$(adjusted p-value)", color=TEXT_COLOR)
        ax.set_title("Functional enrichment of TOP80 non-overlap genes (Enrichr)", color=TEXT_COLOR, pad=10)
        ax.grid(True, axis="x", lw=0.6, color=GRID_COLOR, alpha=0.55)

        savefig(fig, "FIGE_enrichment_dotplot_TOP80_nonoverlap")
        plt.show()
else:
    print("FIG E skipped: missing", ENR_FP)

# ============================================================
# FIG F — Coefficient heatmap (stable TOP80 multinomial LR)
# Enhanced journal-style version
# Keeps same output filename
# ============================================================
if COEF_FP is not None:
    coef = pd.read_csv(COEF_FP, sep=None, engine="python")

    gene_col = "GENE" if "GENE" in coef.columns else coef.columns[0]
    class_cols = [c for c in coef.columns if c != gene_col]

    desired = ["Basal", "Her2", "LumA", "LumB"]
    class_cols = [c for c in desired if c in class_cols] + [c for c in class_cols if c not in desired]

    mat = coef[class_cols].to_numpy(dtype=float)
    score = np.max(np.abs(mat), axis=1)

    topN = int(min(40, coef.shape[0]))
    idx = np.argsort(score)[::-1][:topN]

    coef_top = coef.iloc[idx].copy()
    genes = coef_top[gene_col].astype(str).to_list()
    M = coef_top[class_cols].to_numpy(dtype=float)

    Mz = (M - M.mean(axis=0, keepdims=True)) / (M.std(axis=0, keepdims=True) + 1e-8)

    vmax = np.max(np.abs(Mz))
    vmin = -vmax

    fig, ax = plt.subplots(figsize=(6.8, 8.6))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    im = ax.imshow(
        Mz,
        aspect="auto",
        cmap="RdBu_r",
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest"
    )

    ax.set_title(
        "Stable TOP80 multinomial LR coefficients",
        color=TEXT_COLOR,
        pad=10
    )

    ax.set_xticks(np.arange(len(class_cols)))
    ax.set_xticklabels(class_cols, rotation=25, ha="right", color=TEXT_COLOR)

    ax.set_yticks(np.arange(len(genes)))
    ax.set_yticklabels(genes, fontsize=8.5, color=TEXT_COLOR)

    ax.tick_params(axis="both", length=0)

    for spine in ax.spines.values():
        spine.set_visible(False)

    cbar = fig.colorbar(im, fraction=0.046, pad=0.04)
    cbar.set_label("Z-scored coefficient", color=TEXT_COLOR)
    cbar.ax.tick_params(labelsize=9, colors=TEXT_COLOR)

    savefig(fig, "FIGF_TOP80_coeff_heatmap_top40")
    plt.show()

else:
    print("FIG F skipped: no coefficient CSV found. Looked for:", COEF_CANDIDATES)

print("\n✅ Done. Figures saved in:", FIG_DIR)

In [ ]:
# ============================================================
# FIG0 — Clean high-contrast workflow schematic
# Reduced title-to-figure gap, more vertical room, symmetric layout
# ============================================================

import os
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# -----------------------
# Paths
# -----------------------
DATA_DIR = "/content/drive/MyDrive/EXPORT_FOR_COLAB"
OUT_DIR = os.path.join(DATA_DIR, "ML_RESULTS_UPGRADED")
FIG_DIR = os.path.join(OUT_DIR, "FIGURES_4CLASS_PAPER")
os.makedirs(FIG_DIR, exist_ok=True)

# -----------------------
# Typography
# -----------------------
plt.rcParams.update({
    "figure.dpi": 250,
    "savefig.dpi": 1200,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "font.size": 11,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# -----------------------
# Colors / line styling
# -----------------------
TXT = "#111111"
EDGE = "#000000"
ARROW = "#000000"

FC_BOX   = "#F2F2F2"
FC_MAIN  = "#E8E8E8"
FC_EVAL  = "#EFEFEF"
FC_WHITE = "#FFFFFF"

LW_BOX = 1.8
LW_ARROW = 2.0
LW_DASH = 1.4

# -----------------------
# Canvas
# -----------------------
fig, ax = plt.subplots(figsize=(14.2, 9.0))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

# -----------------------
# Helpers
# -----------------------
def add_box(x, y, w, h, text, fc=FC_BOX, fontsize=10.8, weight="normal",
            ha="center", lw=LW_BOX, rounding=0.004):
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f"round,pad=0.008,rounding_size={rounding}",
        linewidth=lw,
        edgecolor=EDGE,
        facecolor=fc,
        joinstyle="miter"
    )
    ax.add_patch(patch)

    tx = x + w/2 if ha == "center" else x + 0.016
    ax.text(
        tx, y + h/2, text,
        ha=ha, va="center",
        fontsize=fontsize,
        fontweight=weight,
        color=TXT,
        linespacing=1.15,
        wrap=True
    )
    return patch

def add_arrow(x1, y1, x2, y2, lw=LW_ARROW, ms=16, rad=0.0):
    arr = FancyArrowPatch(
        (x1, y1), (x2, y2),
        arrowstyle="-|>",
        mutation_scale=ms,
        linewidth=lw,
        color=ARROW,
        shrinkA=2,
        shrinkB=2,
        connectionstyle=f"arc3,rad={rad}",
        capstyle="butt",
        joinstyle="miter"
    )
    ax.add_patch(arr)

def add_dashed_line(x1, y1, x2, y2, lw=LW_DASH):
    ax.plot(
        [x1, x2], [y1, y2],
        ls=(0, (4, 3)),
        lw=lw,
        color=EDGE,
        solid_capstyle="butt"
    )

def add_section_label(text, x, y):
    ax.text(
        x, y, text,
        ha="left", va="bottom",
        fontsize=13.2,
        fontweight="bold",
        color=TXT
    )

# -----------------------
# Title
# -----------------------
fig.suptitle(
    "Leakage-controlled PAM50 benchmarking framework in METABRIC",
    fontsize=18,
    fontweight="bold",
    y=0.942
)

# -----------------------
# Layout coordinates
# Shifted upward to reduce title gap and free bottom crowding
# -----------------------
top_y  = 0.760
mid_y  = 0.500
arm_y  = 0.275
eval_y = 0.145
out_y  = 0.040

# -----------------------
# Section labels
# -----------------------
add_section_label("Cohort construction", 0.07, 0.875)
add_section_label("Benchmark definition", 0.07, 0.615)
add_section_label("Paired comparison",   0.07, 0.390)
add_section_label("Study outputs",       0.07, 0.125)

# -----------------------
# 1) Cohort construction
# -----------------------
box_w = 0.22
box_h = 0.085

add_box(
    0.10, top_y, box_w, box_h,
    "Raw METABRIC cohort\nExpression profiles\nn = 1,981",
    fc=FC_BOX, fontsize=11.4, weight="bold"
)

add_box(
    0.39, top_y, box_w, box_h,
    "SAMPLE_ID matching\nExpression and clinical table\nn = 1,980",
    fc=FC_BOX, fontsize=11.0
)

add_box(
    0.68, top_y, box_w, box_h,
    "PAM50 harmonization\nExclude nc (n = 6)\nLabeled cohort n = 1,974",
    fc=FC_BOX, fontsize=11.0
)

add_arrow(0.32, top_y + box_h/2, 0.39, top_y + box_h/2)
add_arrow(0.61, top_y + box_h/2, 0.68, top_y + box_h/2)

# -----------------------
# 2) Benchmark definition
# -----------------------
add_box(
    0.08, mid_y, 0.18, 0.085,
    "Secondary analysis\nTumor-only 5-class\nn = 1,826",
    fc=FC_BOX, fontsize=10.6
)

add_box(
    0.35, mid_y - 0.010, 0.30, 0.105,
    "Primary analysis (main claim)\nTumor-only 4-class benchmark\nBasal / Her2 / LumA / LumB\nn = 1,608",
    fc=FC_MAIN, fontsize=12.0, weight="bold"
)

add_box(
    0.74, mid_y, 0.18, 0.085,
    "Exploratory analysis\n6-class setting\nn = 1,974",
    fc=FC_BOX, fontsize=10.6
)

anchor_x = 0.79
anchor_y0 = top_y
anchor_y1 = mid_y + 0.085

add_arrow(anchor_x, anchor_y0, anchor_x, anchor_y1)
add_dashed_line(anchor_x, anchor_y0, 0.50, mid_y + 0.095)
add_dashed_line(anchor_x, anchor_y0, 0.17, mid_y + 0.085)

# -----------------------
# 3) Paired comparison
# -----------------------
arm_w = 0.29
arm_h = 0.105

left_arm_x  = 0.09
right_arm_x = 0.62

add_box(
    left_arm_x, arm_y, arm_w, arm_h,
    "PAM50 reference arm\n"
    "• genefu centroids\n"
    "• gene intersection and alias handling\n"
    "• train-only normalization\n"
    "• nearest-centroid scoring",
    fc=FC_WHITE, fontsize=10.7, ha="left"
)

add_box(
    right_arm_x, arm_y, arm_w, arm_h,
    "Compact supervised arm\n"
    "• train-only preprocessing\n"
    "• L1 feature selection → TOP80 genes\n"
    "• multinomial logistic regression\n"
    "• model tuning on validation data only",
    fc=FC_WHITE, fontsize=10.7, ha="left"
)

primary_bottom_y = mid_y - 0.010
primary_left_x   = 0.42
primary_right_x  = 0.58

left_arm_top_x   = left_arm_x + arm_w/2
right_arm_top_x  = right_arm_x + arm_w/2

add_arrow(primary_left_x, primary_bottom_y, left_arm_top_x, arm_y + arm_h)
add_arrow(primary_right_x, primary_bottom_y, right_arm_top_x, arm_y + arm_h)

# -----------------------
# Paired evaluation
# -----------------------
eval_x, eval_w, eval_h = 0.29, 0.42, 0.078

add_box(
    eval_x, eval_y, eval_w, eval_h,
    "Identical label space and identical train/validation/test splits for both arms\n"
    "Five independent seeds (123–127); paired evaluation on held-out test sets",
    fc=FC_EVAL, fontsize=10.8, weight="bold"
)

eval_left_target_x  = eval_x + 0.13
eval_right_target_x = eval_x + eval_w - 0.13
eval_top_y = eval_y + eval_h

add_arrow(left_arm_x + arm_w/2, arm_y, eval_left_target_x, eval_top_y)
add_arrow(right_arm_x + arm_w/2, arm_y, eval_right_target_x, eval_top_y)

# -----------------------
# 4) Outputs
# -----------------------
out_w = 0.22
out_h = 0.060

add_box(
    0.08, out_y, out_w, out_h,
    "Performance\nmacro-F1, balanced accuracy,\nmacro ROC-AUC, macro PR-AUC",
    fc=FC_BOX, fontsize=10.0
)

add_box(
    0.39, out_y, out_w, out_h,
    "Robustness and stability\nper-seed comparison,\ninclusion frequency, stable core genes",
    fc=FC_BOX, fontsize=10.0
)

add_box(
    0.70, out_y, out_w, out_h,
    "Biological interpretation\noverlap with PAM50,\nenrichment, class-associated genes",
    fc=FC_BOX, fontsize=10.0
)

eval_bottom_y = eval_y
add_arrow(eval_x + 0.08, eval_bottom_y, 0.19, out_y + out_h)
add_arrow(eval_x + eval_w/2, eval_bottom_y, 0.50, out_y + out_h)
add_arrow(eval_x + eval_w - 0.08, eval_bottom_y, 0.81, out_y + out_h)

# -----------------------
# Save
# -----------------------
fig.subplots_adjust(left=0.03, right=0.97, top=0.915, bottom=0.035)

png_fp = os.path.join(FIG_DIR, "FIG0_workflow_clean_journal.png")
pdf_fp = os.path.join(FIG_DIR, "FIG0_workflow_clean_journal.pdf")

fig.savefig(png_fp, bbox_inches="tight", pad_inches=0.08, facecolor="white")
fig.savefig(pdf_fp, bbox_inches="tight", pad_inches=0.08, facecolor="white")

print("Saved:", png_fp)
print("Saved:", pdf_fp)
plt.show()

In [ ]:
#sessionInfo_python.txt
import os, sys, platform, subprocess
import numpy as np
import pandas as pd
import sklearn
import scipy
import matplotlib

release_root = "/content/drive/MyDrive/METABRIC_PAM50_Benchmark_release_v1"
env_dir = os.path.join(release_root, "code", "environment")
os.makedirs(env_dir, exist_ok=True)

py_info_fp = os.path.join(env_dir, "sessionInfo_python.txt")

with open(py_info_fp, "w", encoding="utf-8") as f:
    f.write(f"Python version: {sys.version}\n")
    f.write(f"Platform: {platform.platform()}\n")
    f.write(f"Executable: {sys.executable}\n\n")
    f.write(f"numpy=={np.__version__}\n")
    f.write(f"pandas=={pd.__version__}\n")
    f.write(f"scikit-learn=={sklearn.__version__}\n")
    f.write(f"scipy=={scipy.__version__}\n")
    f.write(f"matplotlib=={matplotlib.__version__}\n")

    try:
        import gseapy
        f.write(f"gseapy=={gseapy.__version__}\n")
    except Exception:
        f.write("gseapy=NOT_INSTALLED\n")

print("Saved:", py_info_fp)

In [ ]:
#a minimal requirements.txt, not the full Colab environment.
import os
from importlib.metadata import version, PackageNotFoundError

release_root = "/content/drive/MyDrive/METABRIC_PAM50_Benchmark_release_v1"
env_dir = os.path.join(release_root, "code", "environment")
os.makedirs(env_dir, exist_ok=True)

req_fp = os.path.join(env_dir, "requirements.txt")

req_pkgs = [
    "numpy",
    "pandas",
    "scikit-learn",
    "scipy",
    "matplotlib",
    "gseapy"
]

lines = []
for pkg in req_pkgs:
    try:
        lines.append(f"{pkg}=={version(pkg)}")
    except PackageNotFoundError:
        pass

with open(req_fp, "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")

print("Saved:", req_fp)
print(open(req_fp, "r", encoding="utf-8").read())

In [ ]:
# Cell — Generate MD5 checksums for processed input files
# Outputs:
# - docs/MD5_checksums.txt
# Computes MD5 for:
# - X_samples_x_genes.tsv.gz
# - y_labels.tsv
# - meta.tsv
import os
import hashlib

release_root = "/content/drive/MyDrive/METABRIC_PAM50_Benchmark_release_v1"
docs_dir = os.path.join(release_root, "docs")
os.makedirs(docs_dir, exist_ok=True)

data_dir = "/content/drive/MyDrive/EXPORT_FOR_COLAB"

files_to_hash = [
    "X_samples_x_genes.tsv.gz",
    "y_labels.tsv",
    "meta.tsv",
]

def md5_file(fp, block=2**20):
    h = hashlib.md5()
    with open(fp, "rb") as f:
        while True:
            b = f.read(block)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

out_fp = os.path.join(docs_dir, "MD5_checksums.txt")

with open(out_fp, "w", encoding="utf-8") as f:
    f.write("MD5 checksums for processed input files\n\n")
    for fn in files_to_hash:
        fp = os.path.join(data_dir, fn)
        if not os.path.exists(fp):
            raise FileNotFoundError(f"Missing file: {fp}")
        md5 = md5_file(fp)
        line = f"{fn}: {md5}"
        print(line)
        f.write(line + "\n")

print("Saved:", out_fp)

In [ ]:
# Cell — Create release README.md for reproducibility package

import os
from textwrap import dedent

release_root = "/content/drive/MyDrive/METABRIC_PAM50_Benchmark_release_v1"
docs_dir = os.path.join(release_root, "docs")
os.makedirs(docs_dir, exist_ok=True)

readme_fp = os.path.join(docs_dir, "README.md")

readme_text = dedent("""
# METABRIC PAM50 Benchmark Release v1

## Project title
Leakage-Controlled PAM50 Benchmarking in METABRIC Reveals a Stable Compact Gene Panel for Intrinsic Subtype Classification

## What the release contains
This release contains the code, processed inputs, main analysis outputs, secondary analysis outputs, manuscript figures, and reproducibility documentation for a leakage-controlled benchmarking workflow for breast cancer intrinsic subtype classification in METABRIC.

The project started in **R** with cohort reconstruction, clinical harmonization, PAM50 label definition, and export of aligned processed inputs, then moved to **Google Colab / Python** for leakage-controlled benchmarking, robustness analyses, compact panel extraction, PAM50 comparison, interpretability analyses, and figure generation.

Release structure:

- `code/R/`
  R code used for cohort reconstruction and export of processed inputs.
- `code/python/`
  Colab notebook or Python script used for benchmarking and figure generation.
- `code/environment/`
  Software environment records for R and Python.
- `data_processed/`
  Processed aligned inputs used by the benchmarking workflow.
- `results_main/`
  Main outputs used in the manuscript primary claim.
- `results_secondary/`
  Secondary and exploratory outputs.
- `figures_main/`
  Main manuscript figures.
- `figures_supplementary/`
  Supplementary figures.
- `docs/`
  Documentation for run order, checksums, and release notes.

## How the workflow runs
The workflow is split into two stages.

First, an **R Markdown** workflow reconstructs the METABRIC cohort from cBioPortal raw files, performs strict `SAMPLE_ID` matching, harmonizes PAM50 labels, and exports aligned processed files into `EXPORT_FOR_COLAB/`.

Second, a **Colab Python** workflow takes these processed inputs, performs leakage-controlled preprocessing and benchmarking, exports PAM50 centroids from `genefu`, evaluates PAM50 and compact logistic-regression models on identical splits, quantifies robustness across seeds, derives a stable compact panel, performs overlap and enrichment analyses, and produces manuscript-ready figures.

## Input files
The main processed input files are:

- `X_samples_x_genes.tsv.gz`
  Processed expression matrix with rows as samples and columns as genes.
- `y_labels.tsv`
  Sample-level labels used for downstream classification tasks.
- `meta.tsv`
  Sample-level metadata including `SAMPLE_ID`, patient identifiers, and label annotations.

These files are created by the R stage and used as the input to the Colab benchmarking stage.

## Run order
Step 1: Run `Transition_to_python.Rmd`
Output: `EXPORT_FOR_COLAB/`

Step 2: Run `METABRIC_PAM50_benchmark.ipynb` or `.py` in Colab
Input: `EXPORT_FOR_COLAB/`
Output: `ML_RESULTS_UPGRADED/`

Step 3: Use the files in `results_main/` and `figures_main/` for the manuscript

## Main outputs used in manuscript
Examples of key manuscript outputs include:

- `cohort_flow_table.tsv`
- `pam50_centroids.tsv`
- `pam50_genes.tsv`
- `pam50_eval_primary4_val_test.csv`
- `compact_eval_primary4_val_test_TOP80.csv`
- `compact_lr_primary4_aucpr_val_test_TOP80.csv`
- `robustness_pam50_vs_compactLR_4class_per_seed.csv`
- `robustness_pam50_vs_compactLR_4class_summary.csv`
- `stable_compact_panel_TOP80_5seeds_5.csv`
- `TOP80_overlap_with_PAM50.csv`
- `TOP80_nonoverlap_vs_PAM50.csv`
- `enrichment_TOP80_nonoverlap_enrichr.csv`
- `STABLE_TOP80_multinomialLR_coefficients_4class_seed123split.csv`
- `STABLE_TOP80_top15_genes_per_class_4class_seed123split.csv`

## How figures map to manuscript figure numbers
Current figure-file mapping is as follows:

- **Figure 1** → `fig1_workflow.pdf`
- **Figure 2** → `fig2_robustness.pdf`
- **Figure 3** → `fig3_confusion.pdf`
- **Figure 4** → `fig4_coefficients_Heatmap.pdf`

If PNG versions are also included in the release, they should use the same base names.

## Notes on manuscript assembly
The current LaTeX manuscript uses the following figure files directly:

- `fig1_workflow.pdf`
- `fig2_robustness.pdf`
- `fig3_confusion.pdf`
- `fig4_coefficients_Heatmap.pdf`

Make sure these filenames are preserved exactly during release packaging and manuscript submission.

## Contact
**Amira Abozaid**
Biotechnology Program, Institute of Basic and Applied Sciences, Egypt-Japan University of Science and Technology (E-JUST)
Molecular Biology Division, Zoology Department, Faculty of Science, Alexandria University
Email: Amira.abozaid@ejust.edu.eg

For questions about the release, code, or manuscript assembly, contact the corresponding author.
""").strip() + "\n"

with open(readme_fp, "w", encoding="utf-8") as f:
    f.write(readme_text)

print("Saved:", readme_fp)

In [ ]:
# Cell — Create docs/FILE_MANIFEST.tsv

import os
import pandas as pd

release_root = "/content/drive/MyDrive/METABRIC_PAM50_Benchmark_release_v1"
docs_dir = os.path.join(release_root, "docs")
os.makedirs(docs_dir, exist_ok=True)

manifest_fp = os.path.join(docs_dir, "FILE_MANIFEST.tsv")

manifest_rows = [
    {
        "file_name": "cohort_flow_table.tsv",
        "manuscript_use": "Table 1 source",
        "notes": "Cohort reconstruction and final subtype counts"
    },
    {
        "file_name": "robustness_pam50_vs_compactLR_4class_summary.csv",
        "manuscript_use": "Table 2 source",
        "notes": "Mean and SD across five seeds for PAM50 versus compact model on the primary 4-class test benchmark"
    },
    {
        "file_name": "robustness_pam50_vs_compactLR_4class_per_seed.csv",
        "manuscript_use": "Results Section 3.2 source",
        "notes": "Per-seed paired benchmark results for the primary 4-class analysis"
    },
    {
        "file_name": "pam50_eval_primary4_val_test.csv",
        "manuscript_use": "Results Section 3.2 source",
        "notes": "Primary PAM50 nearest-centroid benchmark on validation and test splits"
    },
    {
        "file_name": "compact_eval_primary4_val_test_TOP80.csv",
        "manuscript_use": "Results Section 3.2 source",
        "notes": "Compact LR TOP80 validation and test evaluation"
    },
    {
        "file_name": "compact_lr_primary4_aucpr_val_test_TOP80.csv",
        "manuscript_use": "Results Section 3.2 source",
        "notes": "Compact LR macro ROC-AUC and macro PR-AUC values for the primary 4-class task"
    },
    {
        "file_name": "stable_compact_panel_TOP80_5seeds_5.csv",
        "manuscript_use": "Results Section 3.3 source",
        "notes": "Stable TOP80 panel defined by inclusion frequency across five seeds"
    },
    {
        "file_name": "TOP80_overlap_with_PAM50.csv",
        "manuscript_use": "Results Section 3.4 source",
        "notes": "Genes shared between the stable TOP80 panel and PAM50"
    },
    {
        "file_name": "TOP80_nonoverlap_vs_PAM50.csv",
        "manuscript_use": "Results Section 3.4 source",
        "notes": "Stable TOP80 genes not present in PAM50"
    },
    {
        "file_name": "enrichment_TOP80_nonoverlap_enrichr.csv",
        "manuscript_use": "Results Section 3.4 source / Supplementary table",
        "notes": "Functional enrichment results for stable TOP80 genes not overlapping PAM50"
    },
    {
        "file_name": "STABLE_TOP80_multinomialLR_coefficients_4class_seed123split.csv",
        "manuscript_use": "Figure 4 source / Results Section 3.4 source",
        "notes": "Subtype-specific multinomial logistic-regression coefficients for the stable TOP80 panel on the seed-123 primary split"
    },
    {
        "file_name": "STABLE_TOP80_top15_genes_per_class_4class_seed123split.csv",
        "manuscript_use": "Supplementary table",
        "notes": "Top positively weighted genes per subtype from the stable TOP80 model on the seed-123 split"
    },
    {
        "file_name": "pam50_centroids.tsv",
        "manuscript_use": "Methods source",
        "notes": "Exported PAM50 centroids from genefu used for the primary baseline implementation"
    },
    {
        "file_name": "pam50_genes.tsv",
        "manuscript_use": "Methods source",
        "notes": "Canonical PAM50 gene list exported from genefu"
    },

    # Figure files actually generated in Colab
    {
        "file_name": "FIGURES_4CLASS_PAPER/FIG0_workflow_clean_journal.pdf",
        "manuscript_use": "Release figure asset corresponding to workflow overview",
        "notes": "High-resolution workflow schematic generated in Colab"
    },
    {
        "file_name": "FIGURES_4CLASS_PAPER/FIG0_workflow_clean_journal.png",
        "manuscript_use": "Release figure asset corresponding to workflow overview",
        "notes": "PNG version of the workflow schematic"
    },
    {
        "file_name": "FIGURES_4CLASS_PAPER/FIGA_robustness_paired_point_range_meanSD_TEST.png",
        "manuscript_use": "Release figure asset corresponding to robustness comparison",
        "notes": "Paired robustness plot across five seeds"
    },
    {
        "file_name": "FIGURES_4CLASS_PAPER/FIGA_robustness_paired_point_range_meanSD_TEST.pdf",
        "manuscript_use": "Release figure asset corresponding to robustness comparison",
        "notes": "PDF version of the paired robustness plot"
    },
    {
        "file_name": "FIGURES_4CLASS_PAPER/FIGB_CM_PAM50_TEST_seed123.png",
        "manuscript_use": "Release figure asset corresponding to PAM50 confusion matrix",
        "notes": "Row-normalized confusion matrix for PAM50 on the seed-123 test split"
    },
    {
        "file_name": "FIGURES_4CLASS_PAPER/FIGB_CM_PAM50_TEST_seed123.pdf",
        "manuscript_use": "Release figure asset corresponding to PAM50 confusion matrix",
        "notes": "PDF version of the PAM50 seed-123 confusion matrix"
    },
    {
        "file_name": "FIGURES_4CLASS_PAPER/FIGB_CM_CompactLR_TEST_seed123.png",
        "manuscript_use": "Release figure asset corresponding to compact model confusion matrix",
        "notes": "Row-normalized confusion matrix for compact LR on the seed-123 test split"
    },
    {
        "file_name": "FIGURES_4CLASS_PAPER/FIGB_CM_CompactLR_TEST_seed123.pdf",
        "manuscript_use": "Release figure asset corresponding to compact model confusion matrix",
        "notes": "PDF version of the compact LR seed-123 confusion matrix"
    },
    {
        "file_name": "FIGURES_4CLASS_PAPER/FIGB_CM_PAM50_vs_Compact_TEST_seed123_combined.png",
        "manuscript_use": "Release figure asset corresponding to combined confusion matrices",
        "notes": "Combined journal-style side-by-side confusion matrix figure"
    },
    {
        "file_name": "FIGURES_4CLASS_PAPER/FIGB_CM_PAM50_vs_Compact_TEST_seed123_combined.pdf",
        "manuscript_use": "Release figure asset corresponding to combined confusion matrices",
        "notes": "PDF version of the combined confusion matrix figure"
    },
    {
        "file_name": "FIGURES_4CLASS_PAPER/FIGC_macro_ROC_PR_seed123.png",
        "manuscript_use": "Supplementary figure asset",
        "notes": "Macro ROC and PR curves for the seed-123 test split"
    },
    {
        "file_name": "FIGURES_4CLASS_PAPER/FIGC_macro_ROC_PR_seed123.pdf",
        "manuscript_use": "Supplementary figure asset",
        "notes": "PDF version of macro ROC and PR curves for the seed-123 test split"
    },
    {
        "file_name": "FIGURES_4CLASS_PAPER/FIGD_TOP80_overlap_summary.png",
        "manuscript_use": "Supplementary figure asset",
        "notes": "Bar summary of overlap versus non-overlap between stable TOP80 and PAM50"
    },
    {
        "file_name": "FIGURES_4CLASS_PAPER/FIGD_TOP80_overlap_summary.pdf",
        "manuscript_use": "Supplementary figure asset",
        "notes": "PDF version of the overlap summary figure"
    },
    {
        "file_name": "FIGURES_4CLASS_PAPER/FIGE_enrichment_dotplot_TOP80_nonoverlap.png",
        "manuscript_use": "Supplementary figure asset",
        "notes": "Dot plot of enrichment results for stable TOP80 genes not overlapping PAM50"
    },
    {
        "file_name": "FIGURES_4CLASS_PAPER/FIGE_enrichment_dotplot_TOP80_nonoverlap.pdf",
        "manuscript_use": "Supplementary figure asset",
        "notes": "PDF version of the enrichment dot plot"
    },
    {
        "file_name": "FIGURES_4CLASS_PAPER/FIGF_TOP80_coeff_heatmap_top40.png",
        "manuscript_use": "Release figure asset corresponding to coefficient heatmap",
        "notes": "Heatmap of stable TOP80 multinomial logistic-regression coefficients across classes"
    },
    {
        "file_name": "FIGURES_4CLASS_PAPER/FIGF_TOP80_coeff_heatmap_top40.pdf",
        "manuscript_use": "Release figure asset corresponding to coefficient heatmap",
        "notes": "PDF version of the coefficient heatmap"
    },

    # Current manuscript-facing LaTeX names
    {
        "file_name": "fig1_workflow.pdf",
        "manuscript_use": "LaTeX Figure 1",
        "notes": "Current manuscript filename for workflow overview; should point to or be copied from FIG0_workflow_clean_journal.pdf"
    },
    {
        "file_name": "fig2_robustness.pdf",
        "manuscript_use": "LaTeX Figure 2",
        "notes": "Current manuscript filename for robustness figure; should point to or be copied from FIGA_robustness_paired_point_range_meanSD_TEST.pdf"
    },
    {
        "file_name": "fig3_confusion.pdf",
        "manuscript_use": "LaTeX Figure 3",
        "notes": "Current manuscript filename for confusion matrices; should point to or be copied from FIGB_CM_PAM50_vs_Compact_TEST_seed123_combined.pdf"
    },
    {
        "file_name": "fig4_coefficients_Heatmap.pdf",
        "manuscript_use": "LaTeX Figure 4",
        "notes": "Current manuscript filename for coefficient heatmap; should point to or be copied from FIGF_TOP80_coeff_heatmap_top40.pdf"
    },

    # Secondary outputs produced in Colab
    {
        "file_name": "fast_baselines_val_test_secondary_5class.csv",
        "manuscript_use": "Secondary analysis output",
        "notes": "Fast baseline centroid and logistic-regression results for the tumor-only 5-class task"
    },
    {
        "file_name": "panel_gene_scores_secondary_5class.csv",
        "manuscript_use": "Secondary analysis output",
        "notes": "Secondary 5-class selector scores across genes"
    },
    {
        "file_name": "compact_panel_secondary_5class_TOP80.csv",
        "manuscript_use": "Secondary analysis output",
        "notes": "Compact panel genes for the secondary 5-class task"
    },
    {
        "file_name": "compact_panel_val_test_5.csv",
        "manuscript_use": "Secondary analysis output",
        "notes": "Validation and test performance for secondary compact-panel models"
    },
    {
        "file_name": "robustness_5seeds_per_seed_5.csv",
        "manuscript_use": "Secondary analysis output",
        "notes": "Per-seed robustness results for the secondary 5-class task"
    },
    {
        "file_name": "robustness_5seeds_summary_5.csv",
        "manuscript_use": "Secondary analysis output",
        "notes": "Summary robustness results for the secondary 5-class task"
    }
]

manifest_df = pd.DataFrame(manifest_rows, columns=["file_name", "manuscript_use", "notes"])
manifest_df.to_csv(manifest_fp, sep="\t", index=False)

print("Saved:", manifest_fp)
display(manifest_df)